# SMALL V3 — re-evaluate the final model, then try to improve it (every step bootstrapped, B = 10000)

**Part 1 — the original notebook `SMALL_ULTIMATE_V1_DATA_DRIVEN_POCKET`** (Part A–E (V1) + the multi-scale pocket (MS) experiment). Its cells are unchanged except one line
(`[V3-FIX]`: `ATTACH_ESMIF_IF_CACHED = False`, so the V1 screens stay exactly the original ones). Every stage resumes from
Drive (`REUSE` lines): no original model is retrained.

**Part 2 — V3** (search `V3-`):

| cell | what | decision |
|---|---|---|
| V3-2 | current final = 0.6·V1 + 0.4·MS from the **original weights**; Dev must equal 0.409283 exactly | bootstrap CI (Dev / Test300 / Test46 / pooled) |
| V3-3 | new features: **ESM-IF** (512-d, same ESMFold structures) and **ProtT5-XL** (1024-d, computed once ≈10 min) | — |
| V3-4 | Phase A: V1 pocket hypergraph + ESMIF / + ProtT5 / + both (same trainer, same seed) | Dev fold-mean vs V1, margin 0.002; bootstrap vs V1 |
| V3-5 | Phase B: MS with the chosen modality; Phase C: 2 extra seeds per member (seed bagging) | — |
| V3-6 | candidates: current, new-modality pair, bagged pair, union; weights searched on Dev | **cross-fitted Dev** (weights on 4 folds, scored on the 5th), margin 0.002; external + bootstrap vs current |
| V3-7 | zip of predictions / new states / tables → `EXPORT/` | — |

Rules are fixed before running: selection on DevSet1010 only; Test300/46 only confirm. A Dev winner is kept unless the
pooled-test AUPRC CI95 lies entirely below 0 (then the current final stays).

**How to run:** Runtime → GPU (T4 is enough) → **Run all**. Unattended is fine: every new CV fold is saved on Drive, so if
Colab disconnects just **Run all** again. Expected: part 1 ≈ 20–40 min (loading), ProtT5 ≈ 10 min, each new 5-fold CV
≈ 20–40 min (≤ 8 of them), bootstraps ≈ 30–60 min in total.


In [1]:
# ============================================================
# 0. DRIVE / INSTALL / IMPORT
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, ast, math, shutil, subprocess, platform
from collections import Counter, defaultdict

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "numpy", "pandas", "scipy", "scikit-learn",
     "biopython", "tqdm"],
    check=True
)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, Subset
from scipy.spatial import cKDTree
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    f1_score, matthews_corrcoef,
    precision_score, recall_score
)

from Bio.PDB import PDBParser, PDBIO
from Bio.PDB.Structure import Structure
from Bio.PDB.Model import Model
from Bio.PDB.Chain import Chain
from Bio.SeqUtils import seq1
from copy import deepcopy

BASE = Path('/content/drive/MyDrive/Protein-binding-Phi')

ULTIMATE_CANDIDATES = [
    BASE/'GEAR_ULTIMATE_OTHER_BINDINGS',
    BASE/'GEAR_ULTIMATE',
]

ULTIMATE_DIR = next(
    (p for p in ULTIMATE_CANDIDATES if (p/'gear_ultimate').exists()),
    None
)

if ULTIMATE_DIR is None:
    raise FileNotFoundError(
        'GEAR backend not found under GEAR_ULTIMATE_OTHER_BINDINGS or GEAR_ULTIMATE.'
    )

req = ULTIMATE_DIR/'requirements.txt'
if req.exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
        check=True
    )

sys.path.insert(0, str(ULTIMATE_DIR))

try:
    from gear_ultimate import GEAR_ULTIMATE_VERSION as GEAR_VERSION
except Exception:
    from gear_ultimate import __version__ as GEAR_VERSION

from gear_ultimate import gear_core as core
from gear_ultimate import gear_structure as structure
from gear_ultimate import gear_mechanistic as mech
from gear_ultimate import gear_evolution as evo
from gear_ultimate import gear_suite as suite
from gear_ultimate import gear_reporting as reporting

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.set_float32_matmul_precision('high')

RUN_ROOT = BASE/'results'/'SMALL_ULTIMATE_V1_DATA_DRIVEN_POCKET'
RUN_ROOT.mkdir(parents=True, exist_ok=True)

CACHE_ROOT = BASE/'cache'/'small_ultimate_v1'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print('BASE:', BASE)
print('GEAR backend:', ULTIMATE_DIR)
print('GEAR version:', GEAR_VERSION)
print('Device:', DEVICE)
print('Run root:', RUN_ROOT)


Mounted at /content/drive
BASE: /content/drive/MyDrive/Protein-binding-Phi
GEAR backend: /content/drive/MyDrive/Protein-binding-Phi/GEAR_ULTIMATE_OTHER_BINDINGS
GEAR version: 2026.08-other-bindings-v1
Device: cuda
Run root: /content/drive/MyDrive/Protein-binding-Phi/results/SMALL_ULTIMATE_V1_DATA_DRIVEN_POCKET


## 1. Master switches

Default = full small-binding run.

All generated feature blocks, pocket graphs, fold states, and validation predictions are persisted for resume.


In [2]:
# ============================================================
# 1. MASTER SWITCHES
# ============================================================
ESM_BATCH_SIZE = 8

RUN_FEATURE_GENERATION = True
RUN_FAMILY_SCREEN = True
RUN_COMBINATION_LADDER = True
RUN_CUSTOM_ARCHITECTURES = True
RUN_POCKET_HYPERGRAPH = True
RUN_FINAL_EXTERNAL = True

ATTACH_EVO_IF_CACHED = True
ATTACH_ESMIF_IF_CACHED = False   # [V3-FIX] keep the V1 screens identical; V3 adds ESM-IF itself
FORCE_REBUILD_SM_FEATURES = False
FORCE_REBUILD_POCKETS = False

MIN_MEAN_AUPRC_GAIN = 0.002
GLOBAL_COMPLEXITY_MARGIN = 0.002

CUSTOM_BATCH = 4096
CUSTOM_EPOCHS = 45
CUSTOM_PATIENCE = 7
CUSTOM_LR = 3e-4
CUSTOM_WEIGHT_DECAY = 1e-4

POCKET_RADIUS_A = 8.0
POCKET_MAX_PER_PROTEIN = 160
POCKET_PROTEIN_BATCH = 4
POCKET_EPOCHS = 32
POCKET_PATIENCE = 6
POCKET_LR = 3e-4
POCKET_WEIGHT_DECAY = 1e-4
POCKET_LATENT = 192

SEED = 2026

print('Full run:', {
    'features': RUN_FEATURE_GENERATION,
    'family_screen': RUN_FAMILY_SCREEN,
    'combination_ladder': RUN_COMBINATION_LADDER,
    'architectures': RUN_CUSTOM_ARCHITECTURES,
    'pocket_hypergraph': RUN_POCKET_HYPERGRAPH,
    'external': RUN_FINAL_EXTERNAL,
})


Full run: {'features': True, 'family_screen': True, 'combination_ladder': True, 'architectures': True, 'pocket_hypergraph': True, 'external': True}


In [3]:
# ============================================================
# 2. PREPARE / REUSE SMALL-BINDING GEAR CONTEXT
# ============================================================
ctx = suite.prepare_context(
    base=BASE,
    run_version='small_ultimate_v1_context',
    label_column='small_binding',
    models=('MLP',),
    ensure_base=True,
    add_mech=True,
    add_multi=True,
    add_seqctx=True,
    esm_batch_size=ESM_BATCH_SIZE,
    device=DEVICE,
)

display(suite.audit_context(ctx))
display(reporting.dimension_table(ctx.stores['train']))

assert len(ctx.stores['train']) == 1010
assert len(ctx.stores['test300']) == 300
assert len(ctx.stores['test46']) == 46

print('Context sizes: PASS')


[DevSet1010] Reused ESM cache: DevSet1010_ESM2_t33_mean_last2_1280_fp16.pkl
[DevSet1010] Reused structure cache: DevSet1010_spatial_local_v2.pkl
[TestSet300] Reused ESM cache: TestSet300_ESM2_t33_mean_last2_1280_fp16.pkl
[TestSet300] Reused structure cache: TestSet300_spatial_local_v2.pkl
[TestSetNew46] Reused ESM cache: TestSetNew46_ESM2_t33_mean_last2_1280_fp16.pkl
[TestSetNew46] Reused structure cache: TestSetNew46_spatial_local_v2.pkl
[DevSet1010] Reused ATOM cache: DevSet1010_atomic_coordination_v1.pkl
[DevSet1010] Reused CHEM cache: DevSet1010_chemistry_v1_24d.pkl
[TestSet300] Reused ATOM cache: TestSet300_atomic_coordination_v1.pkl
[TestSet300] Reused CHEM cache: TestSet300_chemistry_v1_24d.pkl
[TestSetNew46] Reused ATOM cache: TestSetNew46_atomic_coordination_v1.pkl
[TestSetNew46] Reused CHEM cache: TestSetNew46_chemistry_v1_24d.pkl
[DevSet1010] Reused MECH1 cache: DevSet1010_mechanistic_v1.pkl
[DevSet1010] Reused MULTI2 cache: DevSet1010_multisignal_v2.pkl
[TestSet300] Reused 

,split,proteins,residues,positives,prevalence
0,train,1010,170181,9236,0.054272
1,test300,300,62689,3906,0.062308
2,test46,46,6227,425,0.068251


,block,dim
0,AA,20
6,ACCESS,10
4,ATOM,15
5,CHEM,24
9,CONF,4
11,ELEC3D,8
8,ENV,8
1,ESM,1280
3,LOCAL,8
10,METALSPEC,18


Context sizes: PASS


In [4]:
# ============================================================
# 3. DATAFRAME / OFFICIAL-FOLD AUDIT
# ============================================================
TRAIN_CSV = BASE/'data/development_set/full_grouped_train_binding_sites_df.csv'
TEST300_CSV = BASE/'data/development_set/full_grouped_test_binding_sites_df.csv'
TEST46_CSV = BASE/'data/independent_set/grouped_test_46_new_binding_sites.csv'

def parse_list(v):
    if isinstance(v, str):
        return list(ast.literal_eval(v))
    return list(v)

def load_df(path):
    df = pd.read_csv(path)
    df['prot_id'] = df['prot_id'].astype(str)
    df['sequence'] = df['sequence'].astype(str)

    for c in [
        'metal_binding',
        'nuclear_binding',
        'small_binding',
        'any_ligand_binding_sites',
    ]:
        if c in df.columns:
            df[c] = df[c].apply(parse_list)

    for r in df.itertuples():
        if len(r.sequence) != len(r.small_binding):
            raise RuntimeError(f'{r.prot_id}: small label/sequence mismatch')

    return df

raw_train = load_df(TRAIN_CSV)
test300_df = load_df(TEST300_CSV)
test46_df = load_df(TEST46_CSV)

BASELINE_WORK = BASE/'baseline_reproduction_master_v6'
BN24_CANDIDATES = [
    BASELINE_WORK/'repos'/'bindnode24',
    BASE/'repos'/'bindnode24',
]

BN24 = next((p for p in BN24_CANDIDATES if (p/'.git').exists()), None)

if BN24 is None:
    BN24 = Path('/content/bindnode24_small_v1')
    if not BN24.exists():
        subprocess.run(
            ['git','clone','-q','https://github.com/erckert/bindNode24.git',str(BN24)],
            check=True
        )

fold_root = BN24/'dataset'/'cv_splits'/'predicted_structures'
OFFICIAL_FOLDS = {}
official_order = []
seen = set()

for k in range(1,6):
    ids = [
        x.strip()
        for x in (fold_root/f'ids_split{k}.txt').read_text().splitlines()
        if x.strip()
    ]

    if seen.intersection(ids):
        raise RuntimeError(f'Fold overlap at fold {k}')

    OFFICIAL_FOLDS[k] = ids
    official_order.extend(ids)
    seen.update(ids)

train_lookup = raw_train.set_index('prot_id', drop=False)

missing = set(official_order) - set(raw_train.prot_id)
if missing:
    raise RuntimeError(
        f'Official Dev IDs missing from project CSV: {sorted(missing)[:20]}'
    )

train_df = train_lookup.loc[official_order].reset_index(drop=True)

assert len(train_df) == 1010
assert len(test300_df) == 300
assert len(test46_df) == 46

SPLIT_DF = {
    'DevSet1010': train_df,
    'TestSet300': test300_df,
    'TestSetNew46': test46_df,
}

STORE_KEY = {
    'DevSet1010':'train',
    'TestSet300':'test300',
    'TestSetNew46':'test46',
}

for split,df in SPLIT_DF.items():
    store_ids = set(map(str, ctx.stores[STORE_KEY[split]].keys()))
    csv_ids = set(df.prot_id.astype(str))

    if store_ids != csv_ids:
        raise RuntimeError(
            f'{split}: store/CSV mismatch. '
            f'missing_in_store={sorted(csv_ids-store_ids)[:10]} '
            f'extra_in_store={sorted(store_ids-csv_ids)[:10]}'
        )

print('Exact dataset/fold alignment: PASS')
print('Fold sizes:', {k:len(v) for k,v in OFFICIAL_FOLDS.items()})


Exact dataset/fold alignment: PASS
Fold sizes: {1: 202, 2: 203, 3: 203, 4: 202, 5: 200}


## 4. Optional biological-context reuse

Prior small-binding runs showed that EVO does not consistently improve the small-binding model, so EVO is an optional control, not a core assumption.

ESM-IF is also reuse-only: it is attached only if a complete cache already exists.


In [5]:
# ============================================================
# 4. STORE HELPERS + OPTIONAL EVO / ESMIF
# ============================================================
def get_record(store,pid):
    pid = str(pid)
    if pid in store:
        return store[pid]
    raise KeyError(pid)

def get_block(store,pid,block):
    rec = get_record(store,pid)

    if isinstance(rec,dict):
        if block in rec:
            return np.asarray(rec[block])
        if isinstance(rec.get('features'),dict) and block in rec['features']:
            return np.asarray(rec['features'][block])

    if hasattr(rec,block):
        return np.asarray(getattr(rec,block))

    for attr in ('features','feature_dict','blocks'):
        d = getattr(rec,attr,None)
        if isinstance(d,dict) and block in d:
            return np.asarray(d[block])

    raise KeyError(f'{pid}/{block}')

def set_block(store,pid,block,arr):
    rec = get_record(store,pid)
    arr = np.asarray(arr,dtype=np.float32)

    if isinstance(rec,dict):
        if isinstance(rec.get('features'),dict):
            rec['features'][block] = arr
        else:
            rec[block] = arr
        return

    for attr in ('features','feature_dict','blocks'):
        d = getattr(rec,attr,None)
        if isinstance(d,dict):
            d[block] = arr
            return

    setattr(rec,block,arr)

EVO_AVAILABLE = False

if ATTACH_EVO_IF_CACHED:
    try:
        coverage = evo.attach_evo(
            ctx.stores,
            BASE,
            splits=('train','test300','test46'),
            require_msa=True,
        )
        display(coverage)
        EVO_AVAILABLE = True
        print('EVO48 attached as optional control.')
    except Exception as e:
        print('EVO unavailable/incomplete -> skip:', repr(e))

ESMIF_AVAILABLE = False
ESMIF_CANDIDATE_ROOTS = [
    BASE/'cache'/'gear_metal_v3'/'esmif_v1',
    BASE/'cache'/'nuclear_ultimate_v1'/'esmif_v1',
]

if ATTACH_ESMIF_IF_CACHED:
    for root in ESMIF_CANDIDATE_ROOTS:
        split_dirs = {
            'DevSet1010':root/'DevSet1010'/'esm_if',
            'TestSet300':root/'TestSet300'/'esm_if',
            'TestSetNew46':root/'TestSetNew46'/'esm_if',
        }

        if not all(d.exists() for d in split_dirs.values()):
            continue

        good = True

        for split,df in SPLIT_DF.items():
            d = split_dirs[split]

            for r in df.itertuples():
                fp = d/f'{r.prot_id}.npy'

                if not fp.exists():
                    good = False
                    break

                a = np.load(fp,mmap_mode='r')

                if a.shape != (len(r.sequence),512):
                    good = False
                    break

            if not good:
                break

        if good:
            for split,df in SPLIT_DF.items():
                store = ctx.stores[STORE_KEY[split]]
                d = split_dirs[split]

                for r in df.itertuples():
                    set_block(
                        store,
                        r.prot_id,
                        'ESMIF',
                        np.load(
                            d/f'{r.prot_id}.npy'
                        ).astype(np.float32)
                    )

            ESMIF_AVAILABLE = True
            print('Reused complete ESMIF cache:',root)
            break

print('EVO_AVAILABLE  =',EVO_AVAILABLE)
print('ESMIF_AVAILABLE=',ESMIF_AVAILABLE)


EVO cache:   0%|          | 0/1353 [00:00<?, ?it/s]

,split,protein,msa_depth,has_real_msa
0,train,A0A0F7RDM3,977,True
1,train,A0A0F7RHX8,1024,True
2,train,A0A0H2W6Y8,1023,True
3,train,A0A2B6C3P9,1024,True
4,train,A0A384LKY8,542,True
...,...,...,...,...
1351,test46,Q9KDJ7,1023,True
1352,test46,Q9LFM3,1018,True
1353,test46,Q9SJ89,1022,True
1354,test46,U2EQ00,1021,True


EVO48 attached as optional control.
EVO_AVAILABLE  = True
ESMIF_AVAILABLE= False


# Part A — Small-molecule-specific feature generation

## `SM_SEQCHEM32`
Sequence-local aromatic/hydrophobic/sulfur/flexibility chemistry.

## `SM_POCKET3D44`
3D pocket microenvironment: local packing, shell occupancy, aromatic/hydrophobic/polar composition, tertiary-contact geometry.

## `SM_LIGCHEM20`
Explicit ligand-contact chemistry: H-bond donors/acceptors, aromatic, sulfur, polar, charged and amide/hydroxyl context.

## `SM_TOPO16`
Local spatial graph topology and compactness.

## `SM_AROMSTACK12`
Aromatic-cluster / π-contact proxy. Unlike nuclear, aromatic chemistry is a core hypothesis for small binding because W/F/Y/H enrichment is stable across all three splits.

## `SM_PROTCTX8`
Soft global context only.

Existing `ACCESS`, `TOPO`, `CHEM`, `SOLV`, `ATOM`, and generic structure remain useful controls.


In [6]:
# ============================================================
# 5. SHARED EXACT PDB CACHE — NUCLEAR + SMALL
# ============================================================
from pathlib import Path
import shutil
import tarfile
import json
import hashlib
import os

AA3 = {
    'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
    'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL'
}

# ------------------------------------------------------------
# ONE local cache shared by nuclear + small
# ------------------------------------------------------------
LOCAL_SHARED_ROOT = Path('/content/shared_exact_pdb_v1')

DRIVE_ARCHIVE = (
    BASE / 'cache' / 'shared_exact_pdb_v1.tar.gz'
)

DRIVE_ARCHIVE.parent.mkdir(
    parents=True,
    exist_ok=True
)

# Downstream cells continue using PDB_NORM_ROOT.
PDB_NORM_ROOT = LOCAL_SHARED_ROOT

# Existing work that we should salvage rather than redo.
OLD_NORMALIZED_ROOTS = [
    BASE / 'cache' / 'nuclear_ultimate_v1' / 'normalized_pdb',
    BASE / 'cache' / 'small_ultimate_v1' / 'normalized_pdb',

    BASELINE_WORK
    / 'cache'
    / 'metal_external_normalized_structures',
]

# Raw source roots only used when no normalized copy exists.
RAW_ROOTS = [
    BASE / 'esmFold_pdb_files',
    BASE / 'experimental_pdb',
    BN24 / 'dataset' / 'AlphaFold_predictions',
]

_parser = PDBParser(QUIET=True)


# ============================================================
# helpers
# ============================================================
def standard_residues(chain):
    return [
        r for r in chain
        if (
            r.id[0] == ' '
            and r.resname.strip().upper() in AA3
        )
    ]


def residue_sequence(rr):
    return ''.join(
        seq1(r.resname.strip().upper())
        for r in rr
    )


def pdb_exact_ok(path, target_seq):
    """
    Check whether PDB is already exact chain-A normalized.

    Important:
    use this mainly on /content or on a candidate file,
    NOT recursively over Drive.
    """
    path = Path(path)

    if not path.exists():
        return False

    try:
        st = _parser.get_structure(
            'check',
            str(path)
        )

        model = next(st.get_models())

        if 'A' not in model:
            return False

        rr = standard_residues(
            model['A']
        )

        return (
            residue_sequence(rr)
            == target_seq
        )

    except Exception:
        return False


def normalize_exact_chain_A(src, target_seq, dst):
    """
    Normalize ONLY if we truly do not already have
    an exact-normalized copy.
    """
    st = _parser.get_structure(
        'src',
        str(src)
    )

    model = next(
        st.get_models()
    )

    exact = []
    diag = []

    for ch in model:
        rr = standard_residues(ch)
        seq = residue_sequence(rr)

        diag.append(
            (
                str(ch.id),
                len(seq)
            )
        )

        if seq == target_seq:
            exact.append(
                (
                    ch,
                    rr
                )
            )

    if not exact:
        raise RuntimeError(
            f'{src}: no exact chain match; '
            f'target={len(target_seq)}, '
            f'chains={diag}'
        )

    exact.sort(
        key=lambda x: (
            0
            if str(x[0].id) == 'A'
            else 1,

            str(x[0].id)
        )
    )

    _, rr = exact[0]

    out_st = Structure('normalized')
    out_model = Model(0)
    out_chain = Chain('A')

    out_st.add(out_model)
    out_model.add(out_chain)

    for i, res in enumerate(
        rr,
        start=1
    ):
        r2 = deepcopy(res)
        r2.detach_parent()

        r2.id = (
            ' ',
            i,
            ' '
        )

        out_chain.add(
            r2
        )

    dst.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    io = PDBIO()
    io.set_structure(
        out_st
    )

    io.save(
        str(dst)
    )


# ============================================================
# 1. RESTORE SHARED CACHE IF IT EXISTS
# ============================================================
if (
    DRIVE_ARCHIVE.exists()
    and not LOCAL_SHARED_ROOT.exists()
):
    print(
        'Found shared PDB archive.'
    )

    print(
        'Copy Drive -> /content ...'
    )

    local_tar = (
        Path('/content')
        / DRIVE_ARCHIVE.name
    )

    shutil.copy2(
        DRIVE_ARCHIVE,
        local_tar
    )

    print(
        'Extracting locally ...'
    )

    with tarfile.open(
        local_tar,
        'r:gz'
    ) as tf:
        tf.extractall(
            '/content'
        )

    local_tar.unlink(
        missing_ok=True
    )


LOCAL_SHARED_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. FAST LOOKUP OF OLD NORMALIZED STRUCTURES
# ============================================================
# Direct paths first — no rglob.
def normalized_candidates(
    split,
    pid
):
    pid = str(pid)

    aliases = {
        'DevSet1010': [
            'DevSet1010',
            'train',
        ],

        'TestSet300': [
            'TestSet300',
            'test300',
        ],

        'TestSetNew46': [
            'TestSetNew46',
            'test46',
        ],
    }

    out = []

    for root in OLD_NORMALIZED_ROOTS:
        out.append(
            root / f'{pid}.pdb'
        )

        for alias in aliases.get(
            split,
            []
        ):
            out.append(
                root
                / alias
                / f'{pid}.pdb'
            )

    return out


def raw_candidates(
    pid
):
    pid = str(pid)

    return [
        BASE
        / 'esmFold_pdb_files'
        / f'{pid}.pdb',

        BASE
        / 'experimental_pdb'
        / f'{pid}.pdb',

        BN24
        / 'dataset'
        / 'AlphaFold_predictions'
        / f'{pid}.pdb',
    ]


# Only build fallback index if direct paths actually fail.
RAW_FALLBACK_INDEX = None


def get_raw_fallback_index():
    global RAW_FALLBACK_INDEX

    if RAW_FALLBACK_INDEX is not None:
        return RAW_FALLBACK_INDEX

    print(
        '\nDirect raw paths missed at least one protein.'
    )

    print(
        'Building ONE fallback source index...'
    )

    idx = {}

    for root in RAW_ROOTS:
        if not root.exists():
            continue

        print(
            ' scanning:',
            root
        )

        for dirpath, _, filenames in os.walk(
            root
        ):
            for fn in filenames:
                if not fn.endswith(
                    '.pdb'
                ):
                    continue

                p = (
                    Path(dirpath)
                    / fn
                )

                idx.setdefault(
                    p.stem,
                    p
                )

    RAW_FALLBACK_INDEX = idx

    print(
        'fallback PDBs indexed:',
        len(idx)
    )

    return idx


# ============================================================
# 3. BUILD SHARED LOCAL CACHE
# ============================================================
stats = {
    'already_local': 0,
    'reused_old_normalized': 0,
    'normalized_raw': 0,
}

missing = []

TOTAL = sum(
    len(df)
    for df in SPLIT_DF.values()
)

done = 0


for split, df in SPLIT_DF.items():

    outdir = (
        LOCAL_SHARED_ROOT
        / split
    )

    outdir.mkdir(
        parents=True,
        exist_ok=True
    )

    print(
        '\n',
        split,
        len(df),
        'proteins'
    )

    for r in df.itertuples():

        pid = str(
            r.prot_id
        )

        dst = (
            outdir
            / f'{pid}.pdb'
        )

        # ----------------------------------
        # A. already local shared
        # ----------------------------------
        if pdb_exact_ok(
            dst,
            r.sequence
        ):
            stats[
                'already_local'
            ] += 1

            done += 1
            continue


        # ----------------------------------
        # B. reuse nuclear/small/metal
        #    normalized PDB AS-IS
        # ----------------------------------
        reused = False

        for cand in normalized_candidates(
            split,
            pid
        ):

            if not cand.exists():
                continue

            if pdb_exact_ok(
                cand,
                r.sequence
            ):
                shutil.copy2(
                    cand,
                    dst
                )

                stats[
                    'reused_old_normalized'
                ] += 1

                reused = True
                break


        if reused:
            done += 1

            if done % 100 == 0:
                print(
                    f'  {done}/{TOTAL}',
                    stats
                )

            continue


        # ----------------------------------
        # C. raw source direct lookup
        # ----------------------------------
        src = None

        for cand in raw_candidates(
            pid
        ):
            if cand.exists():
                src = cand
                break


        # ----------------------------------
        # D. last-resort ONE-TIME index
        # ----------------------------------
        if src is None:
            src = get_raw_fallback_index().get(
                pid
            )


        if src is None:
            missing.append(
                (
                    split,
                    pid
                )
            )

            continue


        # ----------------------------------
        # E. normalize only genuinely missing
        # ----------------------------------
        normalize_exact_chain_A(
            src,
            r.sequence,
            dst
        )

        if not pdb_exact_ok(
            dst,
            r.sequence
        ):
            raise RuntimeError(
                f'{split}/{pid}: '
                'normalized PDB failed validation'
            )

        stats[
            'normalized_raw'
        ] += 1

        done += 1


        if done % 100 == 0:
            print(
                f'  {done}/{TOTAL}',
                stats
            )


if missing:
    raise FileNotFoundError(
        f'Missing source PDB for '
        f'{len(missing)} proteins: '
        f'{missing[:20]}'
    )


print(
    '\nCACHE BUILD COMPLETE'
)

print(
    stats
)


# ============================================================
# 4. SAVE ONE ARCHIVE TO DRIVE
# ============================================================
print(
    '\nPacking shared local cache...'
)

local_tar = (
    Path('/content')
    / 'shared_exact_pdb_v1.tar.gz'
)

with tarfile.open(
    local_tar,
    'w:gz'
) as tf:

    tf.add(
        LOCAL_SHARED_ROOT,
        arcname='shared_exact_pdb_v1'
    )


print(
    'Copying ONE archive to Drive...'
)

shutil.copy2(
    local_tar,
    DRIVE_ARCHIVE
)

local_tar.unlink(
    missing_ok=True
)


print(
    '\nExact-sequence shared structures: PASS'
)

print(
    'PDB_NORM_ROOT =',
    PDB_NORM_ROOT
)

print(
    'Persistent archive =',
    DRIVE_ARCHIVE
)

Found shared PDB archive.
Copy Drive -> /content ...
Extracting locally ...


/tmp/ipykernel_723/1301324207.py:237: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(



 DevSet1010 1010 proteins

 TestSet300 300 proteins

 TestSetNew46 46 proteins

CACHE BUILD COMPLETE
{'already_local': 1356, 'reused_old_normalized': 0, 'normalized_raw': 0}

Packing shared local cache...
Copying ONE archive to Drive...

Exact-sequence shared structures: PASS
PDB_NORM_ROOT = /content/shared_exact_pdb_v1
Persistent archive = /content/drive/MyDrive/Protein-binding-Phi/cache/shared_exact_pdb_v1.tar.gz


In [7]:
# ============================================================
# 6. SMALL-BINDING-SPECIFIC FEATURE DEFINITIONS
# ============================================================
FEATURE_CACHE = CACHE_ROOT/'small_explicit_v1'
FEATURE_CACHE.mkdir(parents=True,exist_ok=True)

# Kyte-Doolittle hydropathy.
HYDROPATHY = {
    'A':1.8,'C':2.5,'D':-3.5,'E':-3.5,'F':2.8,
    'G':-0.4,'H':-3.2,'I':4.5,'K':-3.9,'L':3.8,
    'M':1.9,'N':-3.5,'P':-1.6,'Q':-3.5,'R':-4.5,
    'S':-0.8,'T':-0.7,'V':4.2,'W':-0.9,'Y':-1.3,
}

GROUPS = {
    'AROM': set('FWYH'),
    'STRONG_AROM': set('FWY'),
    'SULFUR': set('CM'),
    'HYDRO': set('AVILMFWY'),
    'ALIPHATIC': set('AVILM'),
    'POLAR': set('NQSTY'),
    'ACIDIC': set('DE'),
    'BASIC': set('KRH'),
    'GP': set('GP'),
    'HBD': set('KRHNQSTYW'),
    'HBA': set('DEHNQSTYC'),
    'AMIDE': set('NQ'),
    'HYDROXYL': set('STY'),
}

CHARGE = {
    'K':1.0,
    'R':1.0,
    'H':0.10,
    'D':-1.0,
    'E':-1.0,
}

BACKBONE_ATOMS = {'N','CA','C','O','OXT'}

RING_ATOMS = {
    'PHE':['CG','CD1','CE1','CZ','CE2','CD2'],
    'TYR':['CG','CD1','CE1','CZ','CE2','CD2'],
    'HIS':['CG','ND1','CE1','NE2','CD2'],
    'TRP':['CG','CD1','NE1','CE2','CD2','CE3','CZ3','CH2','CZ2'],
}

def representative_coord(res):
    side = [
        np.asarray(a.coord,dtype=np.float32)
        for a in res
        if a.element != 'H' and a.name not in BACKBONE_ATOMS
    ]

    if side:
        return np.mean(np.stack(side),axis=0)

    if 'CA' in res:
        return np.asarray(res['CA'].coord,dtype=np.float32)

    atoms = [
        np.asarray(a.coord,dtype=np.float32)
        for a in res
    ]

    if not atoms:
        raise RuntimeError('Residue has no atoms')

    return np.mean(np.stack(atoms),axis=0)

def ring_geometry(res):
    rn = res.resname.strip().upper()
    atoms = RING_ATOMS.get(rn)

    if atoms is None:
        return None,None

    cc = [
        np.asarray(res[a].coord,dtype=np.float32)
        for a in atoms
        if a in res
    ]

    if len(cc) < 3:
        return None,None

    x = np.stack(cc)
    centroid = x.mean(axis=0)
    centered = x-centroid[None,:]
    cov = centered.T@centered/max(len(x)-1,1)
    eigval,eigvec = np.linalg.eigh(cov)
    normal = eigvec[:,np.argmin(eigval)]
    normal = normal/(np.linalg.norm(normal)+1e-8)

    return centroid.astype(np.float32),normal.astype(np.float32)

def connected_component_sizes(indices,coords,threshold):
    if len(indices)==0:
        return {}

    idx = np.asarray(indices,dtype=int)
    cc = np.asarray(coords,dtype=np.float32)
    parent = list(range(len(idx)))

    def find(x):
        while parent[x]!=x:
            parent[x]=parent[parent[x]]
            x=parent[x]
        return x

    def union(a,b):
        ra,rb=find(a),find(b)
        if ra!=rb:
            parent[rb]=ra

    if len(idx)>1:
        tree = cKDTree(cc)
        for a,b in tree.query_pairs(threshold):
            union(a,b)

    comp = defaultdict(list)

    for j,ridx in enumerate(idx):
        comp[find(j)].append(int(ridx))

    out={}

    for members in comp.values():
        n=len(members)
        for r in members:
            out[r]=n

    return out

def sequence_chem_block(seq):
    L=len(seq)
    out=np.zeros((L,32),dtype=np.float32)

    hydro=np.asarray(
        [HYDROPATHY.get(a,0.0)/4.5 for a in seq],
        dtype=np.float32
    )

    for i,a in enumerate(seq):
        out[i,0]=hydro[i]
        out[i,1]=float(a in GROUPS['AROM'])
        out[i,2]=float(a in GROUPS['SULFUR'])
        out[i,3]=float(a in GROUPS['HYDRO'])
        out[i,4]=float(a in GROUPS['ACIDIC'])
        out[i,5]=float(a in GROUPS['POLAR'])
        out[i,6]=float(a in GROUPS['GP'])
        out[i,7]=float(a in GROUPS['BASIC'])

        col=8

        for rad in [2,4,7,10]:
            lo=max(0,i-rad)
            hi=min(L,i+rad+1)
            w=seq[lo:hi]
            h=hydro[lo:hi]

            vals=[
                sum(x in GROUPS['AROM'] for x in w)/len(w),
                sum(x in GROUPS['HYDRO'] for x in w)/len(w),
                sum(x in GROUPS['SULFUR'] for x in w)/len(w),
                sum(x in GROUPS['ACIDIC'] for x in w)/len(w),
                sum(x in GROUPS['GP'] for x in w)/len(w),
                float(np.mean(h)),
            ]

            out[i,col:col+6]=vals
            col+=6

    return out

def build_small_3d_blocks(pdb_path,seq):
    st=PDBParser(QUIET=True).get_structure('p',str(pdb_path))
    m=next(st.get_models())
    residues=standard_residues(m['A'])

    if residue_sequence(residues)!=seq:
        raise RuntimeError(f'{pdb_path}: sequence mismatch')

    coords=np.stack(
        [representative_coord(r) for r in residues],
        axis=0
    ).astype(np.float32)

    L=len(seq)
    tree=cKDTree(coords)

    hydro=np.asarray(
        [HYDROPATHY.get(a,0.0)/4.5 for a in seq],
        dtype=np.float32
    )

    q=np.asarray(
        [CHARGE.get(a,0.0) for a in seq],
        dtype=np.float32
    )

    arom_idx=[
        i for i,a in enumerate(seq)
        if a in GROUPS['AROM']
    ]

    hydro_idx=[
        i for i,a in enumerate(seq)
        if a in GROUPS['HYDRO']
    ]

    arom_comp=connected_component_sizes(
        arom_idx,
        coords[arom_idx] if arom_idx else np.zeros((0,3),dtype=np.float32),
        threshold=8.0
    )

    hydro_comp=connected_component_sizes(
        hydro_idx,
        coords[hydro_idx] if hydro_idx else np.zeros((0,3),dtype=np.float32),
        threshold=8.0
    )

    ring_centroid={}
    ring_normal={}

    for i,res in enumerate(residues):
        c,n=ring_geometry(res)

        if c is not None:
            ring_centroid[i]=c
            ring_normal[i]=n

    pocket=np.zeros((L,44),dtype=np.float32)
    ligchem=np.zeros((L,20),dtype=np.float32)
    topo=np.zeros((L,16),dtype=np.float32)
    arom=np.zeros((L,12),dtype=np.float32)

    neigh6=[
        tree.query_ball_point(coords[i],6.0)
        for i in range(L)
    ]
    neigh8=[
        tree.query_ball_point(coords[i],8.0)
        for i in range(L)
    ]
    neigh10=[
        tree.query_ball_point(coords[i],10.0)
        for i in range(L)
    ]

    for i,a in enumerate(seq):
        # ----------------------------------------------------
        # SM_LIGCHEM20: own interaction chemistry
        # ----------------------------------------------------
        ligchem[i,0]=float(a in GROUPS['HBD'])
        ligchem[i,1]=float(a in GROUPS['HBA'])
        ligchem[i,2]=float(a in GROUPS['AROM'])
        ligchem[i,3]=float(a in GROUPS['SULFUR'])
        ligchem[i,4]=float(a in GROUPS['HYDRO'])
        ligchem[i,5]=float(a in GROUPS['POLAR'])
        ligchem[i,6]=float(a in GROUPS['ACIDIC'])
        ligchem[i,7]=float(a in GROUPS['BASIC'])
        ligchem[i,8]=float(a in GROUPS['AMIDE'])
        ligchem[i,9]=float(a in GROUPS['HYDROXYL'])

        c=10

        for radius in [6.0,10.0]:
            ids=np.asarray(
                tree.query_ball_point(coords[i],radius),
                dtype=int
            )

            ids=ids[ids!=i]
            aa=[seq[j] for j in ids]
            den=max(len(aa),1)

            ligchem[i,c:c+5]=[
                sum(x in GROUPS['HBD'] for x in aa)/den,
                sum(x in GROUPS['HBA'] for x in aa)/den,
                sum(x in GROUPS['AROM'] for x in aa)/den,
                sum(x in GROUPS['SULFUR'] for x in aa)/den,
                sum(x in GROUPS['POLAR'] for x in aa)/den,
            ]

            c+=5

        # ----------------------------------------------------
        # SM_POCKET3D44: four radii x eight descriptors
        # ----------------------------------------------------
        c=0

        for radius in [5.0,7.0,9.0,11.0]:
            ids=np.asarray(
                tree.query_ball_point(coords[i],radius),
                dtype=int
            )
            ids=ids[ids!=i]

            if len(ids):
                aa=[seq[j] for j in ids]
                hh=hydro[ids]
                dd=np.linalg.norm(
                    coords[ids]-coords[i][None,:],
                    axis=1
                )

                vals=[
                    min(len(ids),40)/40.0,
                    sum(x in GROUPS['AROM'] for x in aa)/len(aa),
                    sum(x in GROUPS['HYDRO'] for x in aa)/len(aa),
                    sum(x in GROUPS['SULFUR'] for x in aa)/len(aa),
                    sum(x in GROUPS['POLAR'] for x in aa)/len(aa),
                    sum(x in GROUPS['ACIDIC'] for x in aa)/len(aa),
                    float(np.mean(hh)),
                    min(float(np.mean(dd))/12.0,1.0),
                ]
            else:
                vals=[0.0]*8

            pocket[i,c:c+8]=vals
            c+=8

        def nearest(chars):
            ids=[
                j for j,x in enumerate(seq)
                if x in chars and j!=i
            ]

            if not ids:
                return 1.0

            d=np.linalg.norm(
                coords[ids]-coords[i][None,:],
                axis=1
            )

            return min(float(d.min())/20.0,1.0)

        pocket[i,32]=nearest(GROUPS['AROM'])
        pocket[i,33]=nearest(GROUPS['SULFUR'])
        pocket[i,34]=nearest(GROUPS['POLAR'])
        pocket[i,35]=nearest(GROUPS['HYDRO'])

        # Shell occupancy: packing/cavity proxy.
        d_all=np.linalg.norm(
            coords-coords[i][None,:],
            axis=1
        )

        shell_counts=[
            np.sum((d_all>0.1)&(d_all<=4.0)),
            np.sum((d_all>4.0)&(d_all<=6.0)),
            np.sum((d_all>6.0)&(d_all<=8.0)),
            np.sum((d_all>8.0)&(d_all<=10.0)),
        ]

        pocket[i,36:40]=np.clip(
            np.asarray(shell_counts,dtype=np.float32)/20.0,
            0,1
        )

        ids=np.asarray(neigh8[i],dtype=int)
        ids=ids[ids!=i]

        if len(ids)>=3:
            cc=coords[ids]-coords[ids].mean(axis=0,keepdims=True)
            cov=cc.T@cc/max(len(ids)-1,1)
            eig=np.sort(
                np.clip(np.linalg.eigvalsh(cov),0,None)
            )[::-1]

            total=float(eig.sum()+1e-8)

            linearity=(eig[0]-eig[1])/(eig[0]+1e-8)
            planarity=(eig[1]-eig[2])/(eig[0]+1e-8)
            isotropy=eig[2]/total

            pocket[i,40]=np.clip(linearity,0,1)
            pocket[i,41]=np.clip(planarity,0,1)
            pocket[i,42]=np.clip(isotropy,0,1)

            seqsep=np.abs(ids-i)
            pocket[i,43]=float(np.mean(seqsep>10))

        # ----------------------------------------------------
        # SM_TOPO16
        # ----------------------------------------------------
        n6=[j for j in neigh6[i] if j!=i]
        n8=[j for j in neigh8[i] if j!=i]
        n10=[j for j in neigh10[i] if j!=i]

        topo[i,0]=min(len(n6),25)/25.0
        topo[i,1]=min(len(n8),35)/35.0
        topo[i,2]=min(len(n10),45)/45.0

        a8=[j for j in n8 if seq[j] in GROUPS['AROM']]
        h8=[j for j in n8 if seq[j] in GROUPS['HYDRO']]
        s8=[j for j in n8 if seq[j] in GROUPS['SULFUR']]
        p8=[j for j in n8 if seq[j] in GROUPS['POLAR']]
        d8=[j for j in n8 if seq[j] in GROUPS['ACIDIC']]

        topo[i,3]=min(len(a8),12)/12.0
        topo[i,4]=min(len(h8),20)/20.0
        topo[i,5]=min(len(s8),8)/8.0
        topo[i,6]=min(len(p8),20)/20.0
        topo[i,7]=min(len(d8),15)/15.0

        topo[i,8]=min(arom_comp.get(i,0),15)/15.0
        topo[i,9]=min(hydro_comp.get(i,0),25)/25.0

        if len(n8)>=2:
            cc=coords[n8]
            t2=cKDTree(cc)
            edges=len(t2.query_pairs(8.0))
            possible=len(n8)*(len(n8)-1)/2
            topo[i,10]=edges/max(possible,1)

            seqsep=np.abs(np.asarray(n8)-i)
            topo[i,11]=float(np.mean(seqsep>10))
            topo[i,12]=min(float(np.mean(seqsep))/50.0,1.0)

            comp=Counter(seq[j] for j in n8)
            probs=np.asarray(
                [x/len(n8) for x in comp.values()],
                dtype=float
            )
            topo[i,13]=float(
                -np.sum(probs*np.log(probs+1e-12))/math.log(20)
            )

        # inner/outer packing balance
        topo[i,14]=np.tanh(
            (len(n6)-(len(n10)-len(n6)))/10.0
        )

        # hydrophobic/aromatic density vs acidic density
        topo[i,15]=np.tanh(
            (len(h8)+len(a8)-len(d8))/10.0
        )

        # ----------------------------------------------------
        # SM_AROMSTACK12
        # ----------------------------------------------------
        arom[i,0]=float(a in GROUPS['AROM'])
        arom[i,1]=float(a=='W')
        arom[i,2]=float(a=='F')
        arom[i,3]=float(a=='Y')
        arom[i,4]=float(a=='H')

        for col2,radius in zip([5,6,7],[6.0,8.0,10.0]):
            ids=[
                j for j in tree.query_ball_point(coords[i],radius)
                if j!=i
            ]

            arom[i,col2]=(
                sum(seq[j] in GROUPS['AROM'] for j in ids)
                / max(len(ids),1)
            )

        arom[i,8]=nearest(GROUPS['AROM'])
        arom[i,9]=min(arom_comp.get(i,0),15)/15.0

        # Pairwise aromatic orientation coherence in 10 Å neighborhood.
        ar_nei=[
            j for j in neigh10[i]
            if j in ring_normal and j!=i
        ]

        dots=[]

        for a1 in range(len(ar_nei)):
            for a2 in range(a1+1,len(ar_nei)):
                dots.append(
                    abs(float(np.dot(
                        ring_normal[ar_nei[a1]],
                        ring_normal[ar_nei[a2]]
                    )))
                )

        arom[i,10]=float(np.mean(dots)) if dots else 0.0

        if ar_nei:
            cent=np.mean(
                np.stack([ring_centroid[j] for j in ar_nei]),
                axis=0
            )

            arom[i,11]=min(
                float(np.linalg.norm(coords[i]-cent))/20.0,
                1.0
            )

    return pocket,ligchem,topo,arom

def protein_context(seq):
    L=len(seq)

    p=np.array([
        math.log1p(L)/8.0,
        sum(a in GROUPS['AROM'] for a in seq)/L,
        sum(a in GROUPS['HYDRO'] for a in seq)/L,
        sum(a in GROUPS['SULFUR'] for a in seq)/L,
        sum(a in GROUPS['POLAR'] for a in seq)/L,
        sum(a in GROUPS['ACIDIC'] for a in seq)/L,
        sum(a in GROUPS['GP'] for a in seq)/L,
        float(np.mean([
            HYDROPATHY.get(a,0.0)/4.5
            for a in seq
        ])),
    ],dtype=np.float32)

    return np.repeat(
        p[None,:],
        L,
        axis=0
    )


In [8]:
# ============================================================
# 7. GENERATE / CACHE / ATTACH SMALL-SPECIFIC BLOCKS
# ============================================================
NEW_BLOCKS = {
    'SM_SEQCHEM32':32,
    'SM_POCKET3D44':44,
    'SM_LIGCHEM20':20,
    'SM_TOPO16':16,
    'SM_AROMSTACK12':12,
    'SM_PROTCTX8':8,
}

if RUN_FEATURE_GENERATION:
    for split,df in SPLIT_DF.items():
        outdir = FEATURE_CACHE/split
        outdir.mkdir(parents=True,exist_ok=True)

        for n,r in enumerate(df.itertuples(),start=1):
            fp = outdir/f'{r.prot_id}.npz'

            valid=False

            if fp.exists() and not FORCE_REBUILD_SM_FEATURES:
                try:
                    z=np.load(fp)

                    valid=all(
                        z[k].shape==(len(r.sequence),d)
                        for k,d in NEW_BLOCKS.items()
                    )
                except Exception:
                    valid=False

            if not valid:
                seqb=sequence_chem_block(r.sequence)

                pocket,ligchem,topo,arom=build_small_3d_blocks(
                    PDB_NORM_ROOT/split/f'{r.prot_id}.pdb',
                    r.sequence
                )

                prot=protein_context(r.sequence)

                np.savez_compressed(
                    fp,
                    SM_SEQCHEM32=seqb,
                    SM_POCKET3D44=pocket,
                    SM_LIGCHEM20=ligchem,
                    SM_TOPO16=topo,
                    SM_AROMSTACK12=arom,
                    SM_PROTCTX8=prot,
                )

            if n%250==0:
                print(split,n,'/',len(df))

for split,df in SPLIT_DF.items():
    store=ctx.stores[STORE_KEY[split]]
    outdir=FEATURE_CACHE/split

    for r in df.itertuples():
        z=np.load(outdir/f'{r.prot_id}.npz')

        for k,d in NEW_BLOCKS.items():
            a=z[k].astype(np.float32,copy=False)

            if a.shape!=(len(r.sequence),d):
                raise RuntimeError(
                    f'{split}/{r.prot_id}/{k}: {a.shape}'
                )

            set_block(
                store,
                r.prot_id,
                k,
                a
            )

print('New small-binding blocks attached.')

display(
    reporting.dimension_table(
        ctx.stores['train']
    )
)

audit_blocks = [
    'ESM',
    'ACCESS','ATOM','CHEM','TOPO','SOLV',
    'SPATIAL','LOCAL','SEQMOTIF',
    *NEW_BLOCKS.keys(),
]

if EVO_AVAILABLE:
    audit_blocks.append('EVO')

if ESMIF_AVAILABLE:
    audit_blocks.append('ESMIF')

display(
    reporting.validate_no_direct_label_features(
        ctx.stores,
        audit_blocks
    )
)


DevSet1010 250 / 1010
DevSet1010 500 / 1010
DevSet1010 750 / 1010
DevSet1010 1000 / 1010
TestSet300 250 / 300
New small-binding blocks attached.


,block,dim
0,AA,20
6,ACCESS,10
4,ATOM,15
5,CHEM,24
9,CONF,4
11,ELEC3D,8
8,ENV,8
1,ESM,1280
16,EVO,48
3,LOCAL,8


,split,block,missing_proteins,mean_finite_fraction
0,train,ESM,0,1.0
1,train,ACCESS,0,1.0
2,train,ATOM,0,1.0
3,train,CHEM,0,1.0
4,train,TOPO,0,1.0
5,train,SOLV,0,1.0
6,train,SPATIAL,0,1.0
7,train,LOCAL,0,1.0
8,train,SEQMOTIF,0,1.0
9,train,SM_SEQCHEM32,0,1.0


# Part B — Same-MLP feature/mechanism discovery

All representation experiments use the same GEAR MLP and the same official five protein folds.

This separates representation gain from architecture gain.


In [9]:
# ============================================================
# 8. SINGLE-FAMILY MLP SCREEN — DEV ONLY
# ============================================================
FAMILY_GRID = {
    'ESM':['ESM'],

    # Existing audit-supported controls
    'ESM+ACCESS':['ESM','ACCESS'],
    'ESM+TOPO':['ESM','TOPO'],
    'ESM+ATOM':['ESM','ATOM'],
    'ESM+CHEM':['ESM','CHEM'],
    'ESM+SOLV':['ESM','SOLV'],
    'ESM+SEQMOTIF':['ESM','SEQMOTIF'],
    'ESM+Structure':['ESM','SPATIAL','LOCAL'],

    # New small-specific blocks
    'ESM+SM_SEQCHEM':['ESM','SM_SEQCHEM32'],
    'ESM+SM_POCKET3D':['ESM','SM_POCKET3D44'],
    'ESM+SM_LIGCHEM':['ESM','SM_LIGCHEM20'],
    'ESM+SM_TOPO':['ESM','SM_TOPO16'],
    'ESM+SM_AROMSTACK':['ESM','SM_AROMSTACK12'],
    'ESM+SM_PROTCTX':['ESM','SM_PROTCTX8'],
}

if EVO_AVAILABLE:
    FAMILY_GRID['ESM+EVO']=['ESM','EVO']

if ESMIF_AVAILABLE:
    FAMILY_GRID['ESM+ESMIF']=['ESM','ESMIF']

if RUN_FAMILY_SCREEN:
    family_cv,family_folds=suite.run_dev_cv_only(
        ctx,
        FAMILY_GRID,
        models=('MLP',),
        run_version='small_v1_01_family_mlp',
    )

    family_cv.to_csv(
        RUN_ROOT/'family_cv.csv',
        index=False
    )

    family_folds.to_csv(
        RUN_ROOT/'family_folds.csv',
        index=False
    )

else:
    family_cv=pd.read_csv(
        RUN_ROOT/'family_cv.csv'
    )

    family_folds=pd.read_csv(
        RUN_ROOT/'family_folds.csv'
    )

base_score=float(
    family_cv.loc[
        family_cv.representation.eq('ESM'),
        'CV_AUPRC'
    ].iloc[0]
)

family_rank=family_cv.copy()
family_rank['delta_vs_ESM']=family_rank.CV_AUPRC-base_score
family_rank=family_rank.sort_values(
    'CV_AUPRC',
    ascending=False
)

display(family_rank)

family_rank.to_csv(
    RUN_ROOT/'family_ranked.csv',
    index=False
)

print('ESM baseline:',base_score)


REUSE MLP ESM 1
REUSE MLP ESM 2
REUSE MLP ESM 3
REUSE MLP ESM 4
REUSE MLP ESM 5
REUSE MLP ESM+ACCESS 1
REUSE MLP ESM+ACCESS 2
REUSE MLP ESM+ACCESS 3
REUSE MLP ESM+ACCESS 4
REUSE MLP ESM+ACCESS 5
REUSE MLP ESM+TOPO 1
REUSE MLP ESM+TOPO 2
REUSE MLP ESM+TOPO 3
REUSE MLP ESM+TOPO 4
REUSE MLP ESM+TOPO 5
REUSE MLP ESM+ATOM 1
REUSE MLP ESM+ATOM 2
REUSE MLP ESM+ATOM 3
REUSE MLP ESM+ATOM 4
REUSE MLP ESM+ATOM 5
REUSE MLP ESM+CHEM 1
REUSE MLP ESM+CHEM 2
REUSE MLP ESM+CHEM 3
REUSE MLP ESM+CHEM 4
REUSE MLP ESM+CHEM 5
REUSE MLP ESM+SOLV 1
REUSE MLP ESM+SOLV 2
REUSE MLP ESM+SOLV 3
REUSE MLP ESM+SOLV 4
REUSE MLP ESM+SOLV 5
REUSE MLP ESM+SEQMOTIF 1
REUSE MLP ESM+SEQMOTIF 2
REUSE MLP ESM+SEQMOTIF 3
REUSE MLP ESM+SEQMOTIF 4
REUSE MLP ESM+SEQMOTIF 5
REUSE MLP ESM+Structure 1
REUSE MLP ESM+Structure 2
REUSE MLP ESM+Structure 3
REUSE MLP ESM+Structure 4
REUSE MLP ESM+Structure 5
REUSE MLP ESM+SM_SEQCHEM 1
REUSE MLP ESM+SM_SEQCHEM 2
REUSE MLP ESM+SM_SEQCHEM 3
REUSE MLP ESM+SM_SEQCHEM 4
REUSE MLP ESM+SM_SEQCH

,model,representation,CV_AUPRC,CV_AUPRC_SD,CV_AUROC,delta_vs_ESM
8,MLP,ESM+SM_POCKET3D,0.366919,0.026637,0.849958,0.014745
13,MLP,ESM+Structure,0.362243,0.029628,0.846080,0.010070
1,MLP,ESM+ACCESS,0.361371,0.027588,0.851776,0.009197
14,MLP,ESM+TOPO,0.361230,0.031229,0.849949,0.009056
3,MLP,ESM+CHEM,0.360974,0.027618,0.848286,0.008800
9,MLP,ESM+SM_PROTCTX,0.358490,0.025384,0.848983,0.006316
7,MLP,ESM+SM_LIGCHEM,0.357130,0.024493,0.836049,0.004957
11,MLP,ESM+SM_TOPO,0.356375,0.026696,0.847231,0.004201
12,MLP,ESM+SOLV,0.356209,0.024483,0.845769,0.004035
5,MLP,ESM+SEQMOTIF,0.355184,0.020823,0.845086,0.003011


ESM baseline: 0.35217369108912633


In [10]:
# ============================================================
# 9. MECHANISM COMBINATION LADDER — DEV ONLY
# ============================================================
MECHANISM_GRID = {
    'ESM':['ESM'],

    # Audit-derived existing compact control.
    'ESM+ACCESS+TOPO':
        ['ESM','ACCESS','TOPO'],

    # Pocket shape / accessibility.
    'ESM+POCKET_GEOMETRY':
        [
            'ESM',
            'ACCESS',
            'SM_POCKET3D44',
            'SM_TOPO16',
        ],

    # Aromatic + hydrophobic + sulfur microenvironment.
    'ESM+AROM_HYDRO':
        [
            'ESM',
            'SM_SEQCHEM32',
            'SM_AROMSTACK12',
        ],

    # Interaction chemistry / solvation.
    'ESM+LIGAND_CHEMISTRY':
        [
            'ESM',
            'CHEM',
            'SOLV',
            'SM_LIGCHEM20',
        ],

    # Main small-molecule hypothesis.
    'ESM+POCKET_CHEM':
        [
            'ESM',
            'ACCESS',
            'SM_POCKET3D44',
            'SM_TOPO16',
            'SM_SEQCHEM32',
            'SM_AROMSTACK12',
            'SM_LIGCHEM20',
        ],

    'ESM+POCKET_CHEM+SOLV':
        [
            'ESM',
            'ACCESS','SOLV',
            'SM_POCKET3D44','SM_TOPO16',
            'SM_SEQCHEM32','SM_AROMSTACK12',
            'SM_LIGCHEM20',
        ],

    # Existing ATOM/CHEM are transfer controls, not the main ontology.
    'ESM+POCKET_CHEM+ATOM_CHEM_CONTROL':
        [
            'ESM',
            'ACCESS','SOLV',
            'ATOM','CHEM',
            'SM_POCKET3D44','SM_TOPO16',
            'SM_SEQCHEM32','SM_AROMSTACK12',
            'SM_LIGCHEM20',
        ],
}

if EVO_AVAILABLE:
    MECHANISM_GRID['ESM+POCKET_CHEM+EVO_CONTROL']=[
        'ESM',
        'ACCESS',
        'SM_POCKET3D44','SM_TOPO16',
        'SM_SEQCHEM32','SM_AROMSTACK12',
        'SM_LIGCHEM20',
        'EVO',
    ]

if ESMIF_AVAILABLE:
    MECHANISM_GRID['ESM+POCKET_CHEM+ESMIF']=[
        'ESM',
        'ACCESS',
        'SM_POCKET3D44','SM_TOPO16',
        'SM_SEQCHEM32','SM_AROMSTACK12',
        'SM_LIGCHEM20',
        'ESMIF',
    ]

if RUN_COMBINATION_LADDER:
    combo_cv,combo_folds=suite.run_dev_cv_only(
        ctx,
        MECHANISM_GRID,
        models=('MLP',),
        run_version='small_v1_02_mechanism_combo_mlp',
    )

    combo_cv.to_csv(
        RUN_ROOT/'mechanism_combo_cv.csv',
        index=False
    )

    combo_folds.to_csv(
        RUN_ROOT/'mechanism_combo_folds.csv',
        index=False
    )

else:
    combo_cv=pd.read_csv(
        RUN_ROOT/'mechanism_combo_cv.csv'
    )

    combo_folds=pd.read_csv(
        RUN_ROOT/'mechanism_combo_folds.csv'
    )

combo_rank=combo_cv.sort_values(
    'CV_AUPRC',
    ascending=False
).reset_index(drop=True)

display(combo_rank)

BEST_REP_NAME=str(
    combo_rank.iloc[0].representation
)

BEST_REP_BLOCKS=list(
    MECHANISM_GRID[BEST_REP_NAME]
)

BEST_REP_SCORE=float(
    combo_rank.iloc[0].CV_AUPRC
)

print('LOCKED REPRESENTATION:',BEST_REP_NAME)
print('Blocks:',BEST_REP_BLOCKS)
print('Dev CV AUPRC:',BEST_REP_SCORE)

(RUN_ROOT/'locked_representation.json').write_text(
    json.dumps({
        'representation':BEST_REP_NAME,
        'blocks':BEST_REP_BLOCKS,
        'dev_cv_auprc':BEST_REP_SCORE,
        'selection_source':'DevSet1010 only',
    },indent=2)
)


REUSE MLP ESM 1
REUSE MLP ESM 2
REUSE MLP ESM 3
REUSE MLP ESM 4
REUSE MLP ESM 5
REUSE MLP ESM+ACCESS+TOPO 1
REUSE MLP ESM+ACCESS+TOPO 2
REUSE MLP ESM+ACCESS+TOPO 3
REUSE MLP ESM+ACCESS+TOPO 4
REUSE MLP ESM+ACCESS+TOPO 5
REUSE MLP ESM+POCKET_GEOMETRY 1
REUSE MLP ESM+POCKET_GEOMETRY 2
REUSE MLP ESM+POCKET_GEOMETRY 3
REUSE MLP ESM+POCKET_GEOMETRY 4
REUSE MLP ESM+POCKET_GEOMETRY 5
REUSE MLP ESM+AROM_HYDRO 1
REUSE MLP ESM+AROM_HYDRO 2
REUSE MLP ESM+AROM_HYDRO 3
REUSE MLP ESM+AROM_HYDRO 4
REUSE MLP ESM+AROM_HYDRO 5
REUSE MLP ESM+LIGAND_CHEMISTRY 1
REUSE MLP ESM+LIGAND_CHEMISTRY 2
REUSE MLP ESM+LIGAND_CHEMISTRY 3
REUSE MLP ESM+LIGAND_CHEMISTRY 4
REUSE MLP ESM+LIGAND_CHEMISTRY 5
REUSE MLP ESM+POCKET_CHEM 1
REUSE MLP ESM+POCKET_CHEM 2
REUSE MLP ESM+POCKET_CHEM 3
REUSE MLP ESM+POCKET_CHEM 4
REUSE MLP ESM+POCKET_CHEM 5
REUSE MLP ESM+POCKET_CHEM+SOLV 1
REUSE MLP ESM+POCKET_CHEM+SOLV 2
REUSE MLP ESM+POCKET_CHEM+SOLV 3
REUSE MLP ESM+POCKET_CHEM+SOLV 4
REUSE MLP ESM+POCKET_CHEM+SOLV 5
REUSE MLP ESM+P

,model,representation,CV_AUPRC,CV_AUPRC_SD,CV_AUROC
0,MLP,ESM+POCKET_GEOMETRY,0.377708,0.033064,0.849750
1,MLP,ESM+POCKET_CHEM+EVO_CONTROL,0.374193,0.028098,0.856972
2,MLP,ESM+POCKET_CHEM+SOLV,0.374023,0.031997,0.854792
3,MLP,ESM+POCKET_CHEM+ATOM_CHEM_CONTROL,0.372913,0.028648,0.853947
4,MLP,ESM+ACCESS+TOPO,0.372839,0.030957,0.845852
5,MLP,ESM+POCKET_CHEM,0.371453,0.032185,0.852765
6,MLP,ESM+LIGAND_CHEMISTRY,0.362327,0.024956,0.843168
7,MLP,ESM+AROM_HYDRO,0.357675,0.024084,0.848024
8,MLP,ESM,0.352174,0.033973,0.844059


LOCKED REPRESENTATION: ESM+POCKET_GEOMETRY
Blocks: ['ESM', 'ACCESS', 'SM_POCKET3D44', 'SM_TOPO16']
Dev CV AUPRC: 0.3777080103628627


205

# Part C — Multimodal architecture screen

After the feature representation is frozen, the notebook tests whether the way modalities are fused matters.

Semantic modalities:

- **ESM** — sequence PLM
- **POCKET** — accessibility + pocket geometry + local topology
- **CHEM** — H-bond / solvation / explicit ligand-contact chemistry
- **AROM_HYDRO** — aromatic/hydrophobic/sulfur context
- **BIO** — optional EVO / ESM-IF / sequence motif / protein context

Architectures:

- ConcatMLP
- ProjectedConcat
- GatedFusion
- ModalityAttention
- PocketResidualFusion
- LatentLigandMoE

`LatentLigandMoE` tests the plausible heterogeneity of small-molecule pockets without requiring ligand subtype labels.


In [11]:
# ============================================================
# 10. PACK LOCKED REPRESENTATION INTO SEMANTIC MODALITIES
# ============================================================
def modality_spec(blocks):
    blocks=list(blocks)
    spec={}

    if 'ESM' in blocks:
        spec['ESM']=['ESM']

    pocket=[
        b for b in [
            'ACCESS','TOPO',
            'SM_POCKET3D44','SM_TOPO16'
        ]
        if b in blocks
    ]
    if pocket:
        spec['POCKET']=pocket

    chem=[
        b for b in [
            'CHEM','SOLV','ATOM',
            'SM_LIGCHEM20'
        ]
        if b in blocks
    ]
    if chem:
        spec['CHEM']=chem

    arom=[
        b for b in [
            'SM_SEQCHEM32',
            'SM_AROMSTACK12'
        ]
        if b in blocks
    ]
    if arom:
        spec['AROM_HYDRO']=arom

    bio=[
        b for b in [
            'EVO','ESMIF','SEQMOTIF',
            'SM_PROTCTX8','SPATIAL','LOCAL'
        ]
        if b in blocks
    ]
    if bio:
        spec['BIO']=bio

    assigned={x for v in spec.values() for x in v}
    other=[b for b in blocks if b not in assigned]

    if other:
        spec['OTHER']=other

    return spec

MODALITY_SPEC=modality_spec(BEST_REP_BLOCKS)

print('MODALITY_SPEC =',MODALITY_SPEC)

def concat_blocks(store,pid,blocks):
    xs=[
        get_block(store,pid,b).astype(np.float32,copy=False)
        for b in blocks
    ]

    lens={x.shape[0] for x in xs}

    if len(lens)!=1:
        raise RuntimeError(f'{pid}: block length mismatch')

    return np.concatenate(xs,axis=1)

def build_pack(split):
    df=SPLIT_DF[split]
    store=ctx.stores[STORE_KEY[split]]

    mods={k:[] for k in MODALITY_SPEC}
    yy=[]
    pids=[]
    ridx=[]

    for r in df.itertuples():
        L=len(r.sequence)

        for m,blocks in MODALITY_SPEC.items():
            mods[m].append(
                concat_blocks(
                    store,
                    r.prot_id,
                    blocks
                )
            )

        yy.append(
            np.asarray(
                r.small_binding,
                dtype=np.float32
            )
        )

        pids.extend([str(r.prot_id)]*L)
        ridx.extend(range(L))

    mods={
        k:np.concatenate(v,axis=0).astype(
            np.float16 if k in {'ESM','BIO'} else np.float32,
            copy=False
        )
        for k,v in mods.items()
    }

    return {
        'mods':mods,
        'y':np.concatenate(yy).astype(np.float32),
        'pids':np.asarray(pids,dtype=object),
        'ridx':np.asarray(ridx,dtype=np.int32),
    }

PACKS={
    s:build_pack(s)
    for s in SPLIT_DF
}

DIMS={
    k:v.shape[1]
    for k,v in PACKS['DevSet1010']['mods'].items()
}

for split,p in PACKS.items():
    print(
        split,
        {k:v.shape for k,v in p['mods'].items()},
        'y=',p['y'].shape,
        'positive=',int(p['y'].sum())
    )

print('Modality dims:',DIMS)


MODALITY_SPEC = {'ESM': ['ESM'], 'POCKET': ['ACCESS', 'SM_POCKET3D44', 'SM_TOPO16']}
DevSet1010 {'ESM': (170181, 1280), 'POCKET': (170181, 70)} y= (170181,) positive= 9236
TestSet300 {'ESM': (62689, 1280), 'POCKET': (62689, 70)} y= (62689,) positive= 3906
TestSetNew46 {'ESM': (6227, 1280), 'POCKET': (6227, 70)} y= (6227,) positive= 425
Modality dims: {'ESM': 1280, 'POCKET': 70}


In [12]:
# ============================================================
# 11. MULTIMODAL MODELS
# ============================================================
class ConcatMLP(nn.Module):
    def __init__(self,dims,drop=0.2):
        super().__init__()
        self.keys=sorted(dims)
        self.norm=nn.ModuleDict({
            k:nn.LayerNorm(d)
            for k,d in dims.items()
        })

        d=sum(dims.values())

        self.head=nn.Sequential(
            nn.Linear(d,512),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(512,128),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(128,1),
        )

    def forward(self,x):
        z=torch.cat(
            [
                self.norm[k](x[k])
                for k in self.keys
            ],
            dim=-1
        )

        return self.head(z).squeeze(-1)

class ProjectedConcat(nn.Module):
    def __init__(self,dims,latent=192,drop=0.2):
        super().__init__()

        self.keys=sorted(dims)

        self.enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(d,latent),
                nn.GELU(),
                nn.Dropout(drop)
            )
            for k,d in dims.items()
        })

        self.head=nn.Sequential(
            nn.Linear(
                latent*len(dims),
                256
            ),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(256,1)
        )

    def forward(self,x):
        z=torch.cat(
            [
                self.enc[k](x[k])
                for k in self.keys
            ],
            dim=-1
        )

        return self.head(z).squeeze(-1)

class GatedFusion(nn.Module):
    def __init__(self,dims,latent=192,drop=0.2):
        super().__init__()

        self.keys=sorted(dims)

        self.enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(d,latent),
                nn.GELU(),
                nn.Dropout(drop)
            )
            for k,d in dims.items()
        })

        self.gate=nn.ModuleDict({
            k:nn.Linear(latent,1)
            for k in dims
        })

        self.head=nn.Sequential(
            nn.LayerNorm(latent),
            nn.Linear(latent,192),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(192,1)
        )

    def encode(self,x):
        zz=[]
        gg=[]

        for k in self.keys:
            z=self.enc[k](x[k])
            zz.append(z)
            gg.append(self.gate[k](z))

        Z=torch.stack(zz,dim=1)

        G=torch.softmax(
            torch.cat(gg,dim=1),
            dim=1
        ).unsqueeze(-1)

        return (Z*G).sum(dim=1),G.squeeze(-1)

    def forward(self,x):
        h,_=self.encode(x)
        return self.head(h).squeeze(-1)

class ModalityAttention(nn.Module):
    def __init__(self,dims,latent=192,heads=6,drop=0.2):
        super().__init__()

        assert latent%heads==0

        self.keys=sorted(dims)

        self.enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(d,latent),
                nn.GELU(),
                nn.Dropout(drop)
            )
            for k,d in dims.items()
        })

        self.attn=nn.MultiheadAttention(
            latent,
            heads,
            dropout=drop,
            batch_first=True
        )

        self.head=nn.Sequential(
            nn.LayerNorm(latent),
            nn.Linear(latent,192),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(192,1)
        )

    def forward(self,x):
        Z=torch.stack(
            [
                self.enc[k](x[k])
                for k in self.keys
            ],
            dim=1
        )

        A,_=self.attn(
            Z,Z,Z,
            need_weights=False
        )

        h=(Z+A).mean(dim=1)

        return self.head(h).squeeze(-1)

class PocketResidualFusion(nn.Module):
    def __init__(self,dims,latent=192,drop=0.2):
        super().__init__()

        if 'POCKET' not in dims:
            raise ValueError(
                'PocketResidualFusion requires POCKET modality'
            )

        self.base_keys=sorted(
            [
                k for k in dims
                if k!='POCKET'
            ]
        )

        self.base_enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(d,latent),
                nn.GELU(),
                nn.Dropout(drop)
            )
            for k,d in dims.items()
            if k!='POCKET'
        })

        self.pocket_enc=nn.Sequential(
            nn.LayerNorm(dims['POCKET']),
            nn.Linear(dims['POCKET'],latent),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(latent,latent),
        )

        self.base_gate=nn.ModuleDict({
            k:nn.Linear(latent,1)
            for k in self.base_keys
        })

        self.alpha_logit=nn.Parameter(
            torch.tensor(
                -2.944,
                dtype=torch.float32
            )
        )

        self.head=nn.Sequential(
            nn.LayerNorm(latent),
            nn.Linear(latent,192),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(192,1)
        )

    def forward(self,x):
        zz=[]
        gg=[]

        for k in self.base_keys:
            z=self.base_enc[k](x[k])
            zz.append(z)
            gg.append(self.base_gate[k](z))

        Z=torch.stack(zz,dim=1)

        G=torch.softmax(
            torch.cat(gg,dim=1),
            dim=1
        ).unsqueeze(-1)

        h=(Z*G).sum(dim=1)

        pocket=self.pocket_enc(
            x['POCKET']
        )

        alpha=torch.sigmoid(
            self.alpha_logit
        )

        h=h+alpha*pocket

        return self.head(h).squeeze(-1)

class LatentLigandMoE(nn.Module):
    def __init__(self,dims,latent=192,experts=4,drop=0.2):
        super().__init__()

        self.keys=sorted(dims)

        self.enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(d,latent),
                nn.GELU(),
                nn.Dropout(drop)
            )
            for k,d in dims.items()
        })

        self.mod_gate=nn.ModuleDict({
            k:nn.Linear(latent,1)
            for k in dims
        })

        self.shared=nn.Sequential(
            nn.LayerNorm(latent),
            nn.Linear(latent,latent),
            nn.GELU()
        )

        self.experts=nn.ModuleList([
            nn.Sequential(
                nn.Linear(latent,128),
                nn.GELU(),
                nn.Dropout(drop),
                nn.Linear(128,1)
            )
            for _ in range(experts)
        ])

        self.expert_gate=nn.Linear(
            latent,
            experts
        )

    def forward(self,x):
        zz=[]
        gg=[]

        for k in self.keys:
            z=self.enc[k](x[k])
            zz.append(z)
            gg.append(self.mod_gate[k](z))

        Z=torch.stack(zz,dim=1)

        G=torch.softmax(
            torch.cat(gg,dim=1),
            dim=1
        ).unsqueeze(-1)

        h=self.shared(
            (Z*G).sum(dim=1)
        )

        logits=torch.cat(
            [
                e(h)
                for e in self.experts
            ],
            dim=1
        )

        w=torch.softmax(
            self.expert_gate(h),
            dim=1
        )

        return (
            logits*w
        ).sum(dim=1)

ARCH_FACTORY = {
    'ConcatMLP':
        lambda dims:ConcatMLP(dims),

    'ProjectedConcat':
        lambda dims:ProjectedConcat(dims),

    'GatedFusion':
        lambda dims:GatedFusion(dims),

    'ModalityAttention':
        lambda dims:ModalityAttention(dims),

    'LatentLigandMoE':
        lambda dims:LatentLigandMoE(dims),
}

if 'POCKET' in DIMS:
    ARCH_FACTORY[
        'PocketResidualFusion'
    ]=lambda dims:PocketResidualFusion(dims)

print(
    'Architectures:',
    list(ARCH_FACTORY)
)


Architectures: ['ConcatMLP', 'ProjectedConcat', 'GatedFusion', 'ModalityAttention', 'LatentLigandMoE', 'PocketResidualFusion']


In [13]:
# ============================================================
# 12. CUSTOM CV TRAINER — OFFICIAL FOLDS, RESUME SAFE
# ============================================================
class MultiArrayDataset(Dataset):
    def __init__(self,mods,y):
        self.mods=mods
        self.y=y
        self.keys=sorted(mods)

    def __len__(self):
        return len(self.y)

    def __getitem__(self,i):
        x={
            k:torch.from_numpy(
                np.asarray(
                    self.mods[k][i],
                    dtype=np.float32
                )
            )
            for k in self.keys
        }

        return (
            x,
            torch.tensor(
                self.y[i],
                dtype=torch.float32
            )
        )

def collate_multi(batch):
    keys=sorted(batch[0][0])

    x={
        k:torch.stack(
            [b[0][k] for b in batch],
            dim=0
        )
        for k in keys
    }

    y=torch.stack(
        [b[1] for b in batch]
    )

    return x,y

def ap(y,p):
    return float(
        average_precision_score(
            y,p
        )
    )

def predict_flat(
    model,
    dataset,
    indices=None,
    batch_size=8192
):
    ds=(
        dataset
        if indices is None
        else Subset(
            dataset,
            indices.tolist()
            if isinstance(indices,np.ndarray)
            else indices
        )
    )

    loader=DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_multi
    )

    out=[]

    model.eval()

    with torch.no_grad():
        for x,_ in loader:
            x={
                k:v.to(
                    DEVICE,
                    non_blocking=True
                )
                for k,v in x.items()
            }

            out.append(
                torch.sigmoid(
                    model(x)
                )
                .detach()
                .cpu()
                .numpy()
            )

    return np.concatenate(out)

CUSTOM_STATE_ROOT=(
    RUN_ROOT/
    'custom_states'
)

CUSTOM_STATE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

def train_one_arch_fold(
    arch,
    pack,
    train_idx,
    val_idx,
    fold
):
    torch.manual_seed(
        SEED+fold
    )
    np.random.seed(
        SEED+fold
    )

    ds=MultiArrayDataset(
        pack['mods'],
        pack['y']
    )

    train_ds=Subset(
        ds,
        train_idx.tolist()
    )

    model=ARCH_FACTORY[
        arch
    ](DIMS).to(DEVICE)

    ytr=pack['y'][train_idx]

    npos=max(
        float(ytr.sum()),
        1.0
    )

    nneg=max(
        float(len(ytr)-ytr.sum()),
        1.0
    )

    criterion=nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            np.sqrt(nneg/npos),
            dtype=torch.float32,
            device=DEVICE
        )
    )

    opt=torch.optim.AdamW(
        model.parameters(),
        lr=CUSTOM_LR,
        weight_decay=CUSTOM_WEIGHT_DECAY
    )

    loader=DataLoader(
        train_ds,
        batch_size=CUSTOM_BATCH,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_multi
    )

    best_ap=-1.0
    best_state=None
    bad=0

    for epoch in range(
        1,
        CUSTOM_EPOCHS+1
    ):
        model.train()
        losses=[]

        for x,y in loader:
            x={
                k:v.to(
                    DEVICE,
                    non_blocking=True
                )
                for k,v in x.items()
            }

            y=y.to(
                DEVICE,
                non_blocking=True
            )

            opt.zero_grad(
                set_to_none=True
            )

            logits=model(x)
            loss=criterion(
                logits,y
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                5.0
            )

            opt.step()

            losses.append(
                float(
                    loss.detach().cpu()
                )
            )

        vp=predict_flat(
            model,
            ds,
            val_idx
        )

        vy=pack['y'][val_idx]

        val_ap=ap(
            vy,vp
        )

        if val_ap>best_ap+1e-5:
            best_ap=val_ap

            best_state={
                k:v.detach()
                .cpu()
                .clone()
                for k,v
                in model.state_dict().items()
            }

            bad=0

        else:
            bad+=1

        if (
            epoch==1
            or epoch%5==0
        ):
            print(
                arch,
                'fold',fold,
                'epoch',epoch,
                'loss',
                round(
                    float(np.mean(losses)),
                    4
                ),
                'val_AP',
                round(val_ap,5),
                'best',
                round(best_ap,5)
            )

        if bad>=CUSTOM_PATIENCE:
            break

    model.load_state_dict(
        best_state
    )

    return model,best_state

def run_arch_cv(arch):
    pack=PACKS[
        'DevSet1010'
    ]

    pids=pack['pids']
    y=pack['y']

    ds=MultiArrayDataset(
        pack['mods'],
        pack['y']
    )

    oof=np.full(
        len(y),
        np.nan,
        dtype=np.float32
    )

    rows=[]
    states={}

    adir=(
        CUSTOM_STATE_ROOT/
        arch
    )

    adir.mkdir(
        parents=True,
        exist_ok=True
    )

    for fold,val_ids in OFFICIAL_FOLDS.items():
        val_set=set(
            val_ids
        )

        val_idx=np.flatnonzero(
            np.array([
                p in val_set
                for p in pids
            ])
        )

        train_idx=np.flatnonzero(
            np.array([
                p not in val_set
                for p in pids
            ])
        )

        state_fp=adir/f'fold{fold}.pt'
        pred_fp=adir/f'fold{fold}_val.npy'
        meta_fp=adir/f'fold{fold}.json'

        reuse=False

        if (
            state_fp.exists()
            and pred_fp.exists()
            and meta_fp.exists()
        ):
            try:
                meta=json.loads(
                    meta_fp.read_text()
                )

                pp=np.load(
                    pred_fp
                )

                reuse=(
                    meta.get(
                        'representation'
                    )
                    ==BEST_REP_NAME
                    and meta.get(
                        'n_val'
                    )
                    ==len(val_idx)
                    and pp.shape
                    ==(len(val_idx),)
                    and np.all(
                        np.isfinite(pp)
                    )
                )

            except Exception:
                reuse=False

        if reuse:
            print(
                arch,
                'fold',
                fold,
                'REUSE'
            )

            state=torch.load(
                state_fp,
                map_location='cpu'
            )

            pred=np.load(
                pred_fp
            ).astype(
                np.float32
            )

        else:
            print(
                arch,
                'fold',
                fold,
                'TRAIN'
            )

            model,state=(
                train_one_arch_fold(
                    arch,
                    pack,
                    train_idx,
                    val_idx,
                    fold
                )
            )

            pred=predict_flat(
                model,
                ds,
                val_idx
            ).astype(
                np.float32
            )

            torch.save(
                state,
                state_fp
            )

            np.save(
                pred_fp,
                pred
            )

            meta_fp.write_text(
                json.dumps({
                    'representation':
                        BEST_REP_NAME,
                    'blocks':
                        BEST_REP_BLOCKS,
                    'n_val':
                        int(len(val_idx)),
                },indent=2)
            )

            del model

            if DEVICE.type=='cuda':
                torch.cuda.empty_cache()

        states[fold]=state
        oof[val_idx]=pred

        rows.append({
            'architecture':arch,
            'fold':fold,
            'AUPRC':ap(
                y[val_idx],
                pred
            ),
            'n_val':
                len(val_idx),
            'n_positive':
                int(
                    y[val_idx].sum()
                ),
        })

    if not np.all(
        np.isfinite(oof)
    ):
        raise RuntimeError(
            f'{arch}: incomplete OOF'
        )

    f=pd.DataFrame(
        rows
    )

    return (
        {
            'architecture':
                arch,

            'CV_AUPRC_mean':
                float(
                    f.AUPRC.mean()
                ),

            'CV_AUPRC_std':
                float(
                    f.AUPRC.std(
                        ddof=1
                    )
                ),

            'OOF_AUPRC':
                ap(
                    y,
                    oof
                ),
        },
        f,
        oof,
        states
    )


In [14]:
# ============================================================
# 13. RUN MULTIMODAL ARCHITECTURE SCREEN — DEV ONLY
# ============================================================
ARCH_RESULTS={}
ARCH_FOLDS=[]
ARCH_OOF={}
ARCH_STATES={}

if RUN_CUSTOM_ARCHITECTURES:
    for arch in ARCH_FACTORY:
        print(
            '\n'
            +'='*80
        )
        print(
            'ARCH:',
            arch
        )
        print(
            '='*80
        )

        s,f,o,st=run_arch_cv(
            arch
        )

        ARCH_RESULTS[arch]=s
        ARCH_FOLDS.append(f)
        ARCH_OOF[arch]=o
        ARCH_STATES[arch]=st

arch_summary=pd.DataFrame(
    ARCH_RESULTS.values()
).sort_values(
    'CV_AUPRC_mean',
    ascending=False
).reset_index(
    drop=True
)

arch_folds=pd.concat(
    ARCH_FOLDS,
    ignore_index=True
)

arch_summary.to_csv(
    RUN_ROOT/'architecture_cv.csv',
    index=False
)

arch_folds.to_csv(
    RUN_ROOT/'architecture_folds.csv',
    index=False
)

display(
    arch_summary
)

concat_score=float(
    arch_summary.loc[
        arch_summary.architecture.eq(
            'ConcatMLP'
        ),
        'CV_AUPRC_mean'
    ].iloc[0]
)

best_arch=str(
    arch_summary.iloc[0].architecture
)

best_arch_score=float(
    arch_summary.iloc[0].CV_AUPRC_mean
)

LOCKED_ARCH=(
    best_arch
    if (
        best_arch_score
        -concat_score
        >=MIN_MEAN_AUPRC_GAIN
    )
    else 'ConcatMLP'
)

LOCKED_ARCH_SCORE=float(
    arch_summary.loc[
        arch_summary.architecture.eq(
            LOCKED_ARCH
        ),
        'CV_AUPRC_mean'
    ].iloc[0]
)

print(
    'LOCKED ARCH:',
    LOCKED_ARCH,
    LOCKED_ARCH_SCORE
)

(RUN_ROOT/'locked_architecture.json').write_text(
    json.dumps({
        'architecture':
            LOCKED_ARCH,
        'dev_cv_auprc':
            LOCKED_ARCH_SCORE,
        'concat_cv_auprc':
            concat_score,
        'best_raw_architecture':
            best_arch,
        'best_raw_cv_auprc':
            best_arch_score,
    },indent=2)
)



ARCH: ConcatMLP
ConcatMLP fold 1 REUSE
ConcatMLP fold 2 REUSE
ConcatMLP fold 3 REUSE
ConcatMLP fold 4 REUSE
ConcatMLP fold 5 REUSE

ARCH: ProjectedConcat
ProjectedConcat fold 1 REUSE
ProjectedConcat fold 2 REUSE
ProjectedConcat fold 3 REUSE
ProjectedConcat fold 4 REUSE
ProjectedConcat fold 5 REUSE

ARCH: GatedFusion
GatedFusion fold 1 REUSE
GatedFusion fold 2 REUSE
GatedFusion fold 3 REUSE
GatedFusion fold 4 REUSE
GatedFusion fold 5 REUSE

ARCH: ModalityAttention
ModalityAttention fold 1 REUSE
ModalityAttention fold 2 REUSE
ModalityAttention fold 3 REUSE
ModalityAttention fold 4 REUSE
ModalityAttention fold 5 REUSE

ARCH: LatentLigandMoE
LatentLigandMoE fold 1 REUSE
LatentLigandMoE fold 2 REUSE
LatentLigandMoE fold 3 REUSE
LatentLigandMoE fold 4 REUSE
LatentLigandMoE fold 5 REUSE

ARCH: PocketResidualFusion
PocketResidualFusion fold 1 REUSE
PocketResidualFusion fold 2 REUSE
PocketResidualFusion fold 3 REUSE
PocketResidualFusion fold 4 REUSE
PocketResidualFusion fold 5 REUSE


,architecture,CV_AUPRC_mean,CV_AUPRC_std,OOF_AUPRC
0,ConcatMLP,0.377218,0.026619,0.368111
1,PocketResidualFusion,0.373952,0.025417,0.371000
2,GatedFusion,0.373736,0.026525,0.370851
3,ProjectedConcat,0.373322,0.025479,0.370883
4,LatentLigandMoE,0.373089,0.022399,0.362111
5,ModalityAttention,0.369871,0.022821,0.368575


LOCKED ARCH: ConcatMLP 0.37721812239263486


198

# Part D — Small-molecule pocket hypergraph

The generic residue graph is not assumed to be optimal.

V1 constructs candidate pocket microenvironments around residues whose chemistry is consistently enriched in the audit:

```text
W / F / Y / H / C / M
```

All spatial neighbors within 8 Å become members of a candidate pocket.

The graph is **label-independent** and acts as a small residual correction to the strong local multimodal model.

This tests whether higher-order pocket organization adds signal beyond residue-local pocket descriptors.


In [15]:
# ============================================================
# 14. BUILD / CACHE CANDIDATE POCKET HYPERGRAPHS
# ============================================================
POCKET_CACHE=(
    CACHE_ROOT/
    'pocket_hypergraph_v1'
)

POCKET_CACHE.mkdir(
    parents=True,
    exist_ok=True
)

POCKET_SEEDS=set(
    'WFYHCM'
)

def structure_coords(
    split,
    pid,
    seq
):
    path=(
        PDB_NORM_ROOT/
        split/
        f'{pid}.pdb'
    )

    st=PDBParser(
        QUIET=True
    ).get_structure(
        'p',
        str(path)
    )

    m=next(
        st.get_models()
    )

    rr=standard_residues(
        m['A']
    )

    if (
        residue_sequence(rr)
        !=seq
    ):
        raise RuntimeError(
            f'{split}/{pid}: '
            'normalized PDB mismatch'
        )

    return np.stack(
        [
            representative_coord(r)
            for r in rr
        ],
        axis=0
    ).astype(
        np.float32
    )

def pocket_static_features(
    seq,
    coords,
    members
):
    members=np.asarray(
        members,
        dtype=int
    )

    aa=[
        seq[i]
        for i in members
    ]

    cc=coords[members]

    n=len(
        members
    )

    centroid=cc.mean(
        axis=0
    )

    radial=np.linalg.norm(
        cc-centroid[None,:],
        axis=1
    )

    pair=[]

    if n>=2:
        for i in range(n):
            for j in range(i+1,n):
                pair.append(
                    float(
                        np.linalg.norm(
                            cc[i]-cc[j]
                        )
                    )
                )

    pair=np.asarray(
        pair
        if pair
        else [0.0],
        dtype=np.float32
    )

    hydro=np.asarray(
        [
            HYDROPATHY.get(a,0.0)/4.5
            for a in aa
        ],
        dtype=np.float32
    )

    comp=Counter(
        aa
    )

    probs=np.asarray(
        [
            x/n
            for x in comp.values()
        ],
        dtype=np.float32
    )

    entropy=float(
        -np.sum(
            probs*np.log(
                probs+1e-12
            )
        )
        / math.log(20)
    )

    f=np.zeros(
        22,
        dtype=np.float32
    )

    # Composition
    f[0]=min(n,40)/40.0

    f[1]=(
        sum(
            x in GROUPS['AROM']
            for x in aa
        )
        /n
    )

    f[2]=(
        sum(
            x in GROUPS['SULFUR']
            for x in aa
        )
        /n
    )

    f[3]=(
        sum(
            x in GROUPS['HYDRO']
            for x in aa
        )
        /n
    )

    f[4]=(
        sum(
            x in GROUPS['POLAR']
            for x in aa
        )
        /n
    )

    f[5]=(
        sum(
            x in GROUPS['ACIDIC']
            for x in aa
        )
        /n
    )

    f[6]=(
        sum(
            x in GROUPS['HBD']
            for x in aa
        )
        /n
    )

    f[7]=(
        sum(
            x in GROUPS['HBA']
            for x in aa
        )
        /n
    )

    f[8]=float(
        np.mean(hydro)
    )

    f[9]=(
        sum(
            x in GROUPS['GP']
            for x in aa
        )
        /n
    )

    # Geometry
    f[10]=min(
        float(
            radial.mean()
        )/12.0,
        1.0
    )

    f[11]=min(
        float(
            radial.std()
        )/8.0,
        1.0
    )

    f[12]=min(
        float(
            radial.max()
        )/15.0,
        1.0
    )

    f[13]=min(
        float(
            pair.mean()
        )/15.0,
        1.0
    )

    f[14]=min(
        float(
            pair.std()
        )/10.0,
        1.0
    )

    f[15]=min(
        (
            members.max()
            -members.min()
        )/100.0,
        1.0
    )

    f[16]=entropy

    # Tertiary-contact fraction
    if n>=2:
        seqsep=[]

        for i in range(n):
            for j in range(i+1,n):
                seqsep.append(
                    abs(
                        int(members[i])
                        -int(members[j])
                    )
                )

        seqsep=np.asarray(
            seqsep,
            dtype=float
        )

        f[17]=float(
            np.mean(
                seqsep>10
            )
        )

        f[18]=min(
            float(
                seqsep.mean()
            )/50.0,
            1.0
        )

    # PCA shape
    if n>=3:
        centered=(
            cc
            -cc.mean(
                axis=0,
                keepdims=True
            )
        )

        cov=(
            centered.T
            @centered
            /max(n-1,1)
        )

        eig=np.sort(
            np.clip(
                np.linalg.eigvalsh(
                    cov
                ),
                0,
                None
            )
        )[::-1]

        total=float(
            eig.sum()
            +1e-8
        )

        f[19]=np.clip(
            (
                eig[0]-eig[1]
            )/(eig[0]+1e-8),
            0,
            1
        )

        f[20]=np.clip(
            (
                eig[1]-eig[2]
            )/(eig[0]+1e-8),
            0,
            1
        )

        f[21]=np.clip(
            eig[2]/total,
            0,
            1
        )

    return f

def build_candidate_pockets(
    split,
    pid,
    seq
):
    coords=structure_coords(
        split,
        pid,
        seq
    )

    tree=cKDTree(
        coords
    )

    candidates={}

    for seed,a in enumerate(seq):
        if a not in POCKET_SEEDS:
            continue

        members=tuple(
            sorted(
                tree.query_ball_point(
                    coords[seed],
                    POCKET_RADIUS_A
                )
            )
        )

        if len(members)<4:
            continue

        feat=pocket_static_features(
            seq,
            coords,
            members
        )

        # Label-independent pocket plausibility:
        # enriched chemistry + compact local packing
        # with a mild penalty for strongly acidic environments.
        score=(
            1.4*feat[1]
            +0.8*feat[2]
            +0.8*feat[3]
            +0.5*feat[6]
            +0.5*feat[7]
            -0.5*feat[5]
            +0.25*feat[0]
            +0.15*feat[17]
        )

        old=candidates.get(
            members
        )

        if (
            old is None
            or score>old[0]
        ):
            candidates[
                members
            ]=(
                score,
                feat
            )

    ranked=sorted(
        candidates.items(),
        key=lambda kv:
            kv[1][0],
        reverse=True
    )[
        :POCKET_MAX_PER_PROTEIN
    ]

    edge_res=[]
    edge_pocket=[]
    pocket_feat=[]

    for pidx,(
        members,
        (_,feat)
    ) in enumerate(
        ranked
    ):
        pocket_feat.append(
            feat
        )

        for r in members:
            edge_res.append(
                int(r)
            )

            edge_pocket.append(
                int(pidx)
            )

    return (
        np.asarray(
            edge_res,
            dtype=np.int64
        ),
        np.asarray(
            edge_pocket,
            dtype=np.int64
        ),
        np.asarray(
            pocket_feat,
            dtype=np.float32
        ).reshape(
            -1,
            22
        )
    )

POCKET_RAM={}
POCKET_AUDIT=[]

for split,df in SPLIT_DF.items():
    outdir=(
        POCKET_CACHE/
        split
    )

    outdir.mkdir(
        parents=True,
        exist_ok=True
    )

    for n,r in enumerate(
        df.itertuples(),
        start=1
    ):
        fp=(
            outdir/
            f'{r.prot_id}.npz'
        )

        valid=False

        if (
            fp.exists()
            and not FORCE_REBUILD_POCKETS
        ):
            try:
                z=np.load(fp)

                valid=(
                    z['pocket_feat'].ndim
                    ==2
                    and z['pocket_feat'].shape[1]
                    ==22
                    and len(z['edge_res'])
                    ==len(z['edge_pocket'])
                )

            except Exception:
                valid=False

        if not valid:
            er,ep,pf=build_candidate_pockets(
                split,
                str(r.prot_id),
                r.sequence
            )

            np.savez_compressed(
                fp,
                edge_res=er,
                edge_pocket=ep,
                pocket_feat=pf
            )

        z=np.load(fp)

        er=z[
            'edge_res'
        ].astype(
            np.int64
        )

        ep=z[
            'edge_pocket'
        ].astype(
            np.int64
        )

        pf=z[
            'pocket_feat'
        ].astype(
            np.float32
        )

        POCKET_RAM[
            (
                split,
                str(r.prot_id)
            )
        ]=(
            er,
            ep,
            pf
        )

        covered=(
            len(
                np.unique(er)
            )
            if len(er)
            else 0
        )

        POCKET_AUDIT.append({
            'dataset':
                split,

            'prot_id':
                str(r.prot_id),

            'n_residues':
                len(r.sequence),

            'n_candidate_pockets':
                len(pf),

            'n_memberships':
                len(er),

            'covered_residue_fraction':
                covered
                /max(
                    len(r.sequence),
                    1
                ),
        })

        if n%250==0:
            print(
                split,
                n,
                '/',
                len(df)
            )

pocket_audit=pd.DataFrame(
    POCKET_AUDIT
)

pocket_audit.to_csv(
    RUN_ROOT/'pocket_audit.csv',
    index=False
)

display(
    pocket_audit.groupby(
        'dataset'
    ).agg(
        proteins=(
            'prot_id',
            'nunique'
        ),
        mean_pockets=(
            'n_candidate_pockets',
            'mean'
        ),
        median_pockets=(
            'n_candidate_pockets',
            'median'
        ),
        mean_memberships=(
            'n_memberships',
            'mean'
        ),
        mean_covered_fraction=(
            'covered_residue_fraction',
            'mean'
        ),
    ).reset_index()
)

print(
    'Pocket hypergraph cache: PASS'
)


DevSet1010 250 / 1010
DevSet1010 500 / 1010
DevSet1010 750 / 1010
DevSet1010 1000 / 1010
TestSet300 250 / 300


,dataset,proteins,mean_pockets,median_pockets,mean_memberships,mean_covered_fraction
0,DevSet1010,1010,24.930693,22.0,240.274257,0.647951
1,TestSet300,300,32.196667,28.5,324.926667,0.686800
2,TestSetNew46,46,19.130435,17.0,180.673913,0.624250


Pocket hypergraph cache: PASS


In [16]:
# ============================================================
# 15. POCKET-HYPERGRAPH DATASET / MODEL
# ============================================================
class PocketProteinDataset(
    Dataset
):
    def __init__(
        self,
        split,
        df
    ):
        self.split=split

        self.df=df.reset_index(
            drop=True
        )

        self.store=ctx.stores[
            STORE_KEY[split]
        ]

    def __len__(self):
        return len(
            self.df
        )

    def __getitem__(
        self,
        i
    ):
        r=self.df.iloc[
            i
        ]

        pid=str(
            r.prot_id
        )

        mods={
            m:concat_blocks(
                self.store,
                pid,
                blocks
            ).astype(
                np.float32,
                copy=False
            )
            for m,blocks
            in MODALITY_SPEC.items()
        }

        er,ep,pf=POCKET_RAM[
            (
                self.split,
                pid
            )
        ]

        return {
            'pid':
                pid,

            'mods':
                mods,

            'y':
                np.asarray(
                    r.small_binding,
                    dtype=np.float32
                ),

            'edge_res':
                er,

            'edge_pocket':
                ep,

            'pocket_feat':
                pf,
        }

def collate_pocket(
    batch
):
    keys=sorted(
        batch[0]['mods']
    )

    mods={
        k:torch.from_numpy(
            np.concatenate(
                [
                    b['mods'][k]
                    for b in batch
                ],
                axis=0
            )
        ).float()

        for k in keys
    }

    y=torch.from_numpy(
        np.concatenate(
            [
                b['y']
                for b in batch
            ]
        )
    ).float()

    pocket_feat=[]
    edge_res=[]
    edge_pocket=[]

    roff=0
    poff=0

    for b in batch:
        L=len(
            b['y']
        )

        P=len(
            b['pocket_feat']
        )

        if P:
            pocket_feat.append(
                torch.from_numpy(
                    b['pocket_feat']
                ).float()
            )

        if len(
            b['edge_res']
        ):
            edge_res.append(
                torch.from_numpy(
                    b['edge_res']
                ).long()
                +roff
            )

            edge_pocket.append(
                torch.from_numpy(
                    b['edge_pocket']
                ).long()
                +poff
            )

        roff+=L
        poff+=P

    pocket_feat=(
        torch.cat(
            pocket_feat,
            dim=0
        )
        if pocket_feat
        else torch.zeros(
            (0,22),
            dtype=torch.float32
        )
    )

    edge_res=(
        torch.cat(
            edge_res,
            dim=0
        )
        if edge_res
        else torch.zeros(
            (0,),
            dtype=torch.long
        )
    )

    edge_pocket=(
        torch.cat(
            edge_pocket,
            dim=0
        )
        if edge_pocket
        else torch.zeros(
            (0,),
            dtype=torch.long
        )
    )

    return {
        'mods':
            mods,

        'y':
            y,

        'edge_res':
            edge_res,

        'edge_pocket':
            edge_pocket,

        'pocket_feat':
            pocket_feat,
    }

class PocketResidueEncoder(
    nn.Module
):
    def __init__(
        self,
        dims,
        mode='gated',
        latent=192,
        drop=0.2
    ):
        super().__init__()

        self.keys=sorted(
            dims
        )

        self.mode=mode

        self.enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(
                    d,
                    latent
                ),
                nn.GELU(),
                nn.Dropout(
                    drop
                )
            )

            for k,d
            in dims.items()
        })

        if mode=='gated':
            self.gate=nn.ModuleDict({
                k:nn.Linear(
                    latent,
                    1
                )
                for k in dims
            })

        elif mode=='attention':
            self.attn=nn.MultiheadAttention(
                latent,
                6,
                dropout=drop,
                batch_first=True
            )

        else:
            raise ValueError(
                mode
            )

    def forward(
        self,
        x
    ):
        Z=torch.stack(
            [
                self.enc[k](
                    x[k]
                )
                for k in self.keys
            ],
            dim=1
        )

        if self.mode=='gated':
            G=torch.cat(
                [
                    self.gate[k](
                        Z[:,j,:]
                    )
                    for j,k
                    in enumerate(
                        self.keys
                    )
                ],
                dim=1
            )

            G=torch.softmax(
                G,
                dim=1
            ).unsqueeze(
                -1
            )

            return (
                Z*G
            ).sum(
                dim=1
            )

        A,_=self.attn(
            Z,
            Z,
            Z,
            need_weights=False
        )

        return (
            Z+A
        ).mean(
            dim=1
        )

class SmallPocketHypergraph(
    nn.Module
):
    def __init__(
        self,
        dims,
        mode='gated',
        latent=192,
        moe=False,
        experts=4,
        drop=0.2
    ):
        super().__init__()

        self.encoder=PocketResidueEncoder(
            dims,
            mode=mode,
            latent=latent,
            drop=drop
        )

        self.pocket_static=nn.Sequential(
            nn.LayerNorm(
                22
            ),
            nn.Linear(
                22,
                64
            ),
            nn.GELU()
        )

        self.pocket_mlp=nn.Sequential(
            nn.Linear(
                latent*2+64,
                latent
            ),
            nn.GELU(),
            nn.Dropout(
                drop
            ),
            nn.Linear(
                latent,
                latent
            ),
            nn.GELU(),
        )

        self.message_mlp=nn.Sequential(
            nn.LayerNorm(
                latent
            ),
            nn.Linear(
                latent,
                latent
            ),
            nn.GELU(),
            nn.Dropout(
                drop
            ),
            nn.Linear(
                latent,
                latent
            )
        )

        self.alpha_logit=nn.Parameter(
            torch.tensor(
                -2.944,
                dtype=torch.float32
            )
        )

        self.moe=moe

        if not moe:
            self.head=nn.Sequential(
                nn.LayerNorm(
                    latent
                ),
                nn.Linear(
                    latent,
                    128
                ),
                nn.GELU(),
                nn.Dropout(
                    drop
                ),
                nn.Linear(
                    128,
                    1
                )
            )

        else:
            self.experts=nn.ModuleList([
                nn.Sequential(
                    nn.LayerNorm(
                        latent
                    ),
                    nn.Linear(
                        latent,
                        128
                    ),
                    nn.GELU(),
                    nn.Dropout(
                        drop
                    ),
                    nn.Linear(
                        128,
                        1
                    )
                )

                for _ in range(
                    experts
                )
            ])

            self.expert_gate=nn.Linear(
                latent,
                experts
            )

    def index_mean(
        self,
        x,
        index,
        n
    ):
        out=x.new_zeros(
            (
                n,
                x.shape[-1]
            )
        )

        cnt=x.new_zeros(
            (
                n,
                1
            )
        )

        if len(index):
            out.index_add_(
                0,
                index,
                x
            )

            cnt.index_add_(
                0,
                index,
                x.new_ones(
                    (
                        len(index),
                        1
                    )
                )
            )

        return (
            out
            /cnt.clamp_min(
                1.0
            )
        )

    def index_max(
        self,
        x,
        index,
        n
    ):
        if n == 0:
            return x.new_zeros(
                (0, x.shape[-1])
            )

        base = x.new_full(
            (n, x.shape[-1]),
            -1e9
        )

        if len(index):
            idx = index[:, None].expand(
                -1,
                x.shape[-1]
            )

            # IMPORTANT:
            # out-of-place scatter_reduce for safe autograd
            out = base.scatter_reduce(
                0,
                idx,
                x,
                reduce='amax',
                include_self=True
            )
        else:
            out = base

        # IMPORTANT:
        # no inplace masking
        out = torch.where(
            out < -1e8,
            torch.zeros_like(out),
            out
        )

        return out

    def forward(
        self,
        mods,
        edge_res,
        edge_pocket,
        pocket_feat
    ):
        h=self.encoder(
            mods
        )

        nres=h.shape[0]
        npocket=pocket_feat.shape[0]

        alpha=torch.sigmoid(
            self.alpha_logit
        )

        if (
            npocket
            and edge_res.numel()
        ):
            mean=self.index_mean(
                h[edge_res],
                edge_pocket,
                npocket
            )

            mx=self.index_max(
                h[edge_res],
                edge_pocket,
                npocket
            )

            ps=self.pocket_static(
                pocket_feat
            )

            hp=self.pocket_mlp(
                torch.cat(
                    [
                        mean,
                        mx,
                        ps
                    ],
                    dim=-1
                )
            )

            msg=self.index_mean(
                hp[edge_pocket],
                edge_res,
                nres
            )

            incident=h.new_zeros(
                (
                    nres,
                    1
                )
            )

            incident.index_add_(
                0,
                edge_res,
                h.new_ones(
                    (
                        len(edge_res),
                        1
                    )
                )
            )

            delta=self.message_mlp(
                msg
            )

            delta=(
                delta
                *(
                    incident>0
                ).float()
            )

            h=(
                h
                +alpha*delta
            )

        if not self.moe:
            return self.head(
                h
            ).squeeze(
                -1
            )

        logits=torch.cat(
            [
                e(h)
                for e in self.experts
            ],
            dim=1
        )

        w=torch.softmax(
            self.expert_gate(
                h
            ),
            dim=1
        )

        return (
            logits*w
        ).sum(
            dim=1
        )

POCKET_FACTORY = {
    'SM_PocketHG_Gated':
        lambda dims:
            SmallPocketHypergraph(
                dims,
                mode='gated',
                latent=POCKET_LATENT,
                moe=False
            ),

    'SM_PocketHG_Attention':
        lambda dims:
            SmallPocketHypergraph(
                dims,
                mode='attention',
                latent=POCKET_LATENT,
                moe=False
            ),

    'SM_PocketHG_GatedMoE':
        lambda dims:
            SmallPocketHypergraph(
                dims,
                mode='gated',
                latent=POCKET_LATENT,
                moe=True,
                experts=4
            ),
}

print(
    'Pocket-hypergraph architectures:',
    list(
        POCKET_FACTORY
    )
)


Pocket-hypergraph architectures: ['SM_PocketHG_Gated', 'SM_PocketHG_Attention', 'SM_PocketHG_GatedMoE']


In [17]:
# ============================================================
# 16. POCKET-HYPERGRAPH CV TRAINER — RESUME SAFE
# ============================================================
POCKET_STATE_ROOT=(
    RUN_ROOT/
    'pocket_states'
)

POCKET_STATE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

def move_pocket_batch(
    b
):
    return {
        'mods':{
            k:v.to(
                DEVICE,
                non_blocking=True
            )
            for k,v
            in b['mods'].items()
        },

        'y':
            b['y'].to(
                DEVICE,
                non_blocking=True
            ),

        'edge_res':
            b['edge_res'].to(
                DEVICE,
                non_blocking=True
            ),

        'edge_pocket':
            b['edge_pocket'].to(
                DEVICE,
                non_blocking=True
            ),

        'pocket_feat':
            b['pocket_feat'].to(
                DEVICE,
                non_blocking=True
            ),
    }

def predict_pocket(
    model,
    dataset
):
    loader=DataLoader(
        dataset,
        batch_size=POCKET_PROTEIN_BATCH,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_pocket
    )

    pp=[]
    yy=[]

    model.eval()

    with torch.no_grad():
        for b in loader:
            b=move_pocket_batch(
                b
            )

            p=torch.sigmoid(
                model(
                    b['mods'],
                    b['edge_res'],
                    b['edge_pocket'],
                    b['pocket_feat']
                )
            )

            pp.append(
                p.detach()
                .cpu()
                .numpy()
            )

            yy.append(
                b['y']
                .detach()
                .cpu()
                .numpy()
            )

    return (
        np.concatenate(
            pp
        ),
        np.concatenate(
            yy
        )
    )

def train_pocket_fold(
    arch,
    tr_ds,
    va_ds,
    fold
):
    torch.manual_seed(
        SEED+100+fold
    )

    np.random.seed(
        SEED+100+fold
    )

    model=POCKET_FACTORY[
        arch
    ](DIMS).to(
        DEVICE
    )

    ytr=np.concatenate(
        tr_ds.df.small_binding.apply(
            lambda x:
                np.asarray(
                    x,
                    dtype=np.float32
                )
        ).tolist()
    )

    npos=max(
        float(
            ytr.sum()
        ),
        1.0
    )

    nneg=max(
        float(
            len(ytr)-ytr.sum()
        ),
        1.0
    )

    criterion=nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            np.sqrt(
                nneg/npos
            ),
            dtype=torch.float32,
            device=DEVICE
        )
    )

    opt=torch.optim.AdamW(
        model.parameters(),
        lr=POCKET_LR,
        weight_decay=POCKET_WEIGHT_DECAY
    )

    loader=DataLoader(
        tr_ds,
        batch_size=POCKET_PROTEIN_BATCH,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_pocket
    )

    best_ap=-1.0
    best_state=None
    bad=0

    for epoch in range(
        1,
        POCKET_EPOCHS+1
    ):
        model.train()
        losses=[]

        for b in loader:
            b=move_pocket_batch(
                b
            )

            opt.zero_grad(
                set_to_none=True
            )

            logits=model(
                b['mods'],
                b['edge_res'],
                b['edge_pocket'],
                b['pocket_feat']
            )

            loss=criterion(
                logits,
                b['y']
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                5.0
            )

            opt.step()

            losses.append(
                float(
                    loss.detach()
                    .cpu()
                )
            )

        vp,vy=predict_pocket(
            model,
            va_ds
        )

        val_ap=ap(
            vy,
            vp
        )

        if (
            val_ap
            >best_ap+1e-5
        ):
            best_ap=val_ap

            best_state={
                k:v.detach()
                .cpu()
                .clone()
                for k,v
                in model.state_dict().items()
            }

            bad=0

        else:
            bad+=1

        if (
            epoch==1
            or epoch%5==0
        ):
            print(
                arch,
                'fold',
                fold,
                'epoch',
                epoch,
                'loss',
                round(
                    float(
                        np.mean(
                            losses
                        )
                    ),
                    4
                ),
                'val_AP',
                round(
                    val_ap,
                    5
                ),
                'best',
                round(
                    best_ap,
                    5
                ),
                'alpha',
                round(
                    float(
                        torch.sigmoid(
                            model.alpha_logit
                        )
                        .detach()
                        .cpu()
                    ),
                    4
                )
            )

        if bad>=POCKET_PATIENCE:
            break

    model.load_state_dict(
        best_state
    )

    return (
        model,
        best_state
    )

def run_pocket_cv(
    arch
):
    offsets={}
    cursor=0

    for r in train_df.itertuples():
        L=len(
            r.sequence
        )

        offsets[
            str(
                r.prot_id
            )
        ]=(
            cursor,
            cursor+L
        )

        cursor+=L

    oof=np.full(
        cursor,
        np.nan,
        dtype=np.float32
    )

    rows=[]
    states={}

    adir=(
        POCKET_STATE_ROOT/
        arch
    )

    adir.mkdir(
        parents=True,
        exist_ok=True
    )

    for fold,val_ids in OFFICIAL_FOLDS.items():
        val_set=set(
            val_ids
        )

        tr_df=train_df[
            ~train_df.prot_id.isin(
                val_set
            )
        ].copy()

        va_df=train_df[
            train_df.prot_id.isin(
                val_set
            )
        ].copy()

        tr_ds=PocketProteinDataset(
            'DevSet1010',
            tr_df
        )

        va_ds=PocketProteinDataset(
            'DevSet1010',
            va_df
        )

        state_fp=(
            adir/
            f'fold{fold}.pt'
        )

        pred_fp=(
            adir/
            f'fold{fold}_val.npy'
        )

        meta_fp=(
            adir/
            f'fold{fold}.json'
        )

        expected=int(
            va_df.sequence
            .str.len()
            .sum()
        )

        reuse=False

        if (
            state_fp.exists()
            and pred_fp.exists()
            and meta_fp.exists()
        ):
            try:
                meta=json.loads(
                    meta_fp.read_text()
                )

                pp=np.load(
                    pred_fp
                )

                reuse=(
                    meta.get(
                        'representation'
                    )
                    ==BEST_REP_NAME
                    and meta.get(
                        'n_val'
                    )
                    ==expected
                    and pp.shape
                    ==(expected,)
                    and np.all(
                        np.isfinite(
                            pp
                        )
                    )
                )

            except Exception:
                reuse=False

        if reuse:
            print(
                arch,
                'fold',
                fold,
                'REUSE'
            )

            state=torch.load(
                state_fp,
                map_location='cpu'
            )

            pred=np.load(
                pred_fp
            ).astype(
                np.float32
            )

        else:
            print(
                arch,
                'fold',
                fold,
                'TRAIN'
            )

            model,state=(
                train_pocket_fold(
                    arch,
                    tr_ds,
                    va_ds,
                    fold
                )
            )

            pred,_=predict_pocket(
                model,
                va_ds
            )

            pred=pred.astype(
                np.float32
            )

            torch.save(
                state,
                state_fp
            )

            np.save(
                pred_fp,
                pred
            )

            meta_fp.write_text(
                json.dumps({
                    'representation':
                        BEST_REP_NAME,

                    'blocks':
                        BEST_REP_BLOCKS,

                    'n_val':
                        expected,
                },indent=2)
            )

            del model

            if DEVICE.type=='cuda':
                torch.cuda.empty_cache()

        states[
            fold
        ]=state

        pos=0

        for r in va_df.itertuples():
            L=len(
                r.sequence
            )

            a,b=offsets[
                str(
                    r.prot_id
                )
            ]

            oof[
                a:b
            ]=pred[
                pos:
                pos+L
            ]

            pos+=L

        yv=np.concatenate(
            va_df.small_binding.apply(
                lambda x:
                    np.asarray(
                        x,
                        dtype=np.float32
                    )
            ).tolist()
        )

        rows.append({
            'architecture':
                arch,

            'fold':
                fold,

            'AUPRC':
                ap(
                    yv,
                    pred
                ),

            'n_val':
                len(
                    pred
                ),

            'n_positive':
                int(
                    yv.sum()
                ),
        })

    if not np.all(
        np.isfinite(
            oof
        )
    ):
        raise RuntimeError(
            f'{arch}: incomplete pocket OOF'
        )

    f=pd.DataFrame(
        rows
    )

    yall=np.concatenate(
        train_df.small_binding.apply(
            lambda x:
                np.asarray(
                    x,
                    dtype=np.float32
                )
        ).tolist()
    )

    return (
        {
            'architecture':
                arch,

            'CV_AUPRC_mean':
                float(
                    f.AUPRC.mean()
                ),

            'CV_AUPRC_std':
                float(
                    f.AUPRC.std(
                        ddof=1
                    )
                ),

            'OOF_AUPRC':
                ap(
                    yall,
                    oof
                ),
        },
        f,
        oof,
        states
    )


In [18]:
# ============================================================
# 17. RUN POCKET-HYPERGRAPH SCREEN — DEV ONLY
# ============================================================
POCKET_RESULTS={}
POCKET_FOLDS=[]
POCKET_OOF={}
POCKET_STATES={}

if RUN_POCKET_HYPERGRAPH:
    for arch in POCKET_FACTORY:
        print(
            '\n'
            +'='*80
        )

        print(
            'POCKET HYPERGRAPH:',
            arch
        )

        print(
            '='*80
        )

        s,f,o,st=run_pocket_cv(
            arch
        )

        POCKET_RESULTS[
            arch
        ]=s

        POCKET_FOLDS.append(
            f
        )

        POCKET_OOF[
            arch
        ]=o

        POCKET_STATES[
            arch
        ]=st

pocket_summary=pd.DataFrame(
    POCKET_RESULTS.values()
).sort_values(
    'CV_AUPRC_mean',
    ascending=False
).reset_index(
    drop=True
)

pocket_folds=pd.concat(
    POCKET_FOLDS,
    ignore_index=True
)

pocket_summary.to_csv(
    RUN_ROOT/
    'pocket_hypergraph_cv.csv',
    index=False
)

pocket_folds.to_csv(
    RUN_ROOT/
    'pocket_hypergraph_folds.csv',
    index=False
)

display(
    pocket_summary
)

BEST_POCKET=str(
    pocket_summary.iloc[0]
    .architecture
)

BEST_POCKET_SCORE=float(
    pocket_summary.iloc[0]
    .CV_AUPRC_mean
)

alphas=[]

for fold,state in POCKET_STATES[
    BEST_POCKET
].items():
    if (
        'alpha_logit'
        in state
    ):
        alphas.append(
            float(
                torch.sigmoid(
                    state[
                        'alpha_logit'
                    ]
                ).item()
            )
        )

print(
    'BEST POCKET MODEL:',
    BEST_POCKET
)

print(
    'Dev CV AUPRC:',
    BEST_POCKET_SCORE
)

print(
    'Residual alpha:',
    alphas
)

print(
    'Mean alpha:',
    float(
        np.mean(
            alphas
        )
    )
    if alphas
    else None
)

(RUN_ROOT/'locked_pocket_model.json').write_text(
    json.dumps({
        'architecture':
            BEST_POCKET,

        'dev_cv_auprc':
            BEST_POCKET_SCORE,

        'alpha_by_fold':
            alphas,

        'alpha_mean':
            float(
                np.mean(
                    alphas
                )
            )
            if alphas
            else None,
    },indent=2)
)



POCKET HYPERGRAPH: SM_PocketHG_Gated
SM_PocketHG_Gated fold 1 REUSE
SM_PocketHG_Gated fold 2 REUSE
SM_PocketHG_Gated fold 3 REUSE
SM_PocketHG_Gated fold 4 REUSE
SM_PocketHG_Gated fold 5 REUSE

POCKET HYPERGRAPH: SM_PocketHG_Attention
SM_PocketHG_Attention fold 1 REUSE
SM_PocketHG_Attention fold 2 REUSE
SM_PocketHG_Attention fold 3 REUSE
SM_PocketHG_Attention fold 4 REUSE
SM_PocketHG_Attention fold 5 REUSE

POCKET HYPERGRAPH: SM_PocketHG_GatedMoE
SM_PocketHG_GatedMoE fold 1 REUSE
SM_PocketHG_GatedMoE fold 2 REUSE
SM_PocketHG_GatedMoE fold 3 REUSE
SM_PocketHG_GatedMoE fold 4 REUSE
SM_PocketHG_GatedMoE fold 5 REUSE


,architecture,CV_AUPRC_mean,CV_AUPRC_std,OOF_AUPRC
0,SM_PocketHG_Attention,0.397647,0.040608,0.376346
1,SM_PocketHG_Gated,0.394450,0.038383,0.385554
2,SM_PocketHG_GatedMoE,0.391164,0.028219,0.372161


BEST POCKET MODEL: SM_PocketHG_Attention
Dev CV AUPRC: 0.39764655317751607
Residual alpha: [0.04977594316005707, 0.04997347295284271, 0.04953764006495476, 0.05000319331884384, 0.04984140023589134]
Mean alpha: 0.04982632994651794


271

# Part E — Global Dev-only lock

The final headline family is selected before external evaluation.

The more complex pocket-hypergraph replaces the best local multimodal model only if it improves mean Dev CV AUPRC by at least `GLOBAL_COMPLEXITY_MARGIN`.


In [19]:
# ============================================================
# 18. GLOBAL DEV-ONLY LOCK
# ============================================================
candidates=pd.DataFrame([
    {
        'family':
            'LOCAL_MULTIMODAL',

        'model':
            LOCKED_ARCH,

        'Dev_CV_AUPRC':
            LOCKED_ARCH_SCORE,

        'complexity_rank':
            1,
    },

    {
        'family':
            'POCKET_HYPERGRAPH',

        'model':
            BEST_POCKET,

        'Dev_CV_AUPRC':
            BEST_POCKET_SCORE,

        'complexity_rank':
            2,
    },
])

winner=(
    candidates
    .sort_values(
        'complexity_rank'
    )
    .iloc[0]
    .to_dict()
)

for _,r in (
    candidates
    .sort_values(
        'complexity_rank'
    )
    .iloc[1:]
    .iterrows()
):
    if (
        float(
            r.Dev_CV_AUPRC
        )
        -float(
            winner[
                'Dev_CV_AUPRC'
            ]
        )
        >=GLOBAL_COMPLEXITY_MARGIN
    ):
        winner=r.to_dict()

GLOBAL_FAMILY=str(
    winner[
        'family'
    ]
)

GLOBAL_MODEL=str(
    winner[
        'model'
    ]
)

GLOBAL_SCORE=float(
    winner[
        'Dev_CV_AUPRC'
    ]
)

display(
    candidates
)

print(
    'GLOBAL LOCK:',
    GLOBAL_FAMILY,
    GLOBAL_MODEL,
    GLOBAL_SCORE
)

(RUN_ROOT/'GLOBAL_LOCK.json').write_text(
    json.dumps({
        'family':
            GLOBAL_FAMILY,

        'model':
            GLOBAL_MODEL,

        'dev_cv_auprc':
            GLOBAL_SCORE,

        'complexity_margin':
            GLOBAL_COMPLEXITY_MARGIN,
    },indent=2)
)


,family,model,Dev_CV_AUPRC,complexity_rank
0,LOCAL_MULTIMODAL,ConcatMLP,0.377218,1
1,POCKET_HYPERGRAPH,SM_PocketHG_Attention,0.397647,2


GLOBAL LOCK: POCKET_HYPERGRAPH SM_PocketHG_Attention 0.39764655317751607


140

In [20]:
# ============================================================
# 19. METRIC HELPERS + DEV OOF THRESHOLDS
# ============================================================
def binary_metrics(
    y,
    p,
    threshold
):
    y=np.asarray(
        y,
        dtype=int
    )

    p=np.asarray(
        p,
        dtype=float
    )

    pred=(
        p>=threshold
    ).astype(
        int
    )

    return {
        'AUPRC':
            float(
                average_precision_score(
                    y,p
                )
            ),

        'AUROC':
            float(
                roc_auc_score(
                    y,p
                )
            ),

        'precision':
            float(
                precision_score(
                    y,pred,
                    zero_division=0
                )
            ),

        'recall':
            float(
                recall_score(
                    y,pred,
                    zero_division=0
                )
            ),

        'F1':
            float(
                f1_score(
                    y,pred,
                    zero_division=0
                )
            ),

        'MCC':
            float(
                matthews_corrcoef(
                    y,pred
                )
            ),
    }

def best_mcc_threshold(
    y,
    p
):
    best=(-999,None)

    for t in np.linspace(
        0.01,
        0.99,
        199
    ):
        m=matthews_corrcoef(
            y,
            (
                p>=t
            ).astype(
                int
            )
        )

        if m>best[0]:
            best=(
                m,
                float(t)
            )

    return (
        best[1],
        best[0]
    )

dev_y=PACKS[
    'DevSet1010'
]['y'].astype(
    int
)

LOCAL_OOF=ARCH_OOF[
    LOCKED_ARCH
]

LOCAL_THR,LOCAL_OOF_MCC=(
    best_mcc_threshold(
        dev_y,
        LOCAL_OOF
    )
)

POCKET_OOF_LOCK=POCKET_OOF[
    BEST_POCKET
]

POCKET_THR,POCKET_OOF_MCC=(
    best_mcc_threshold(
        dev_y,
        POCKET_OOF_LOCK
    )
)

print(
    'Local threshold:',
    LOCAL_THR,
    'OOF MCC:',
    LOCAL_OOF_MCC
)

print(
    'Pocket threshold:',
    POCKET_THR,
    'OOF MCC:',
    POCKET_OOF_MCC
)


Local threshold: 0.5593939393939393 OOF MCC: 0.3750286199352274
Pocket threshold: 0.6088888888888889 OOF MCC: 0.3799406947088853


In [21]:
# ============================================================
# 20. FINAL EXTERNAL EVALUATION — LOCKED FAMILY WINNERS
# ============================================================
def predict_local_external(
    split
):
    pack=PACKS[
        split
    ]

    ds=MultiArrayDataset(
        pack['mods'],
        pack['y']
    )

    preds=[]

    for fold,state in ARCH_STATES[
        LOCKED_ARCH
    ].items():
        model=ARCH_FACTORY[
            LOCKED_ARCH
        ](DIMS).to(
            DEVICE
        )

        model.load_state_dict(
            state
        )

        preds.append(
            predict_flat(
                model,
                ds
            )
        )

        del model

        if DEVICE.type=='cuda':
            torch.cuda.empty_cache()

    return np.mean(
        np.stack(
            preds
        ),
        axis=0
    )

def predict_pocket_external(
    split
):
    ds=PocketProteinDataset(
        split,
        SPLIT_DF[
            split
        ]
    )

    preds=[]

    for fold,state in POCKET_STATES[
        BEST_POCKET
    ].items():
        model=POCKET_FACTORY[
            BEST_POCKET
        ](DIMS).to(
            DEVICE
        )

        model.load_state_dict(
            state
        )

        p,_=predict_pocket(
            model,
            ds
        )

        preds.append(
            p
        )

        del model

        if DEVICE.type=='cuda':
            torch.cuda.empty_cache()

    return np.mean(
        np.stack(
            preds
        ),
        axis=0
    )

rows=[]

if RUN_FINAL_EXTERNAL:
    for split in [
        'TestSet300',
        'TestSetNew46'
    ]:
        y=PACKS[
            split
        ]['y'].astype(
            int
        )

        lp=predict_local_external(
            split
        )

        pp=predict_pocket_external(
            split
        )

        for (
            family,
            model,
            p,
            thr,
            devscore
        ) in [
            (
                'LOCAL_MULTIMODAL',
                LOCKED_ARCH,
                lp,
                LOCAL_THR,
                LOCKED_ARCH_SCORE
            ),
            (
                'POCKET_HYPERGRAPH',
                BEST_POCKET,
                pp,
                POCKET_THR,
                BEST_POCKET_SCORE
            ),
        ]:
            m05=binary_metrics(
                y,
                p,
                0.5
            )

            moo=binary_metrics(
                y,
                p,
                thr
            )

            row={
                'family':
                    family,

                'model':
                    model,

                'dataset':
                    split,

                'Dev_CV_AUPRC':
                    devscore,

                'Dev_OOF_MCC_threshold':
                    thr,
            }

            for k,v in m05.items():
                row[
                    f'{k}_fixed0.5'
                ]=v

            for k,v in moo.items():
                row[
                    f'{k}_OOFthr'
                ]=v

            rows.append(
                row
            )

        pd.DataFrame({
            'prot_id':
                PACKS[split]['pids'],

            'residue_index':
                PACKS[split]['ridx'],

            'y_true':
                y,

            'local_probability':
                lp,

            'pocket_probability':
                pp,
        }).to_csv(
            RUN_ROOT/
            f'predictions_{split}.csv.gz',
            index=False
        )

external_df=pd.DataFrame(
    rows
)

external_df.to_csv(
    RUN_ROOT/
    'external_family_winners.csv',
    index=False
)

display(
    external_df
)

headline=external_df[
    external_df.family.eq(
        GLOBAL_FAMILY
    )
    &
    external_df.model.eq(
        GLOBAL_MODEL
    )
].copy()

headline.to_csv(
    RUN_ROOT/
    'FINAL_HEADLINE_EXTERNAL.csv',
    index=False
)

print(
    'HEADLINE:'
)

display(
    headline
)


,family,model,dataset,Dev_CV_AUPRC,Dev_OOF_MCC_threshold,AUPRC_fixed0.5,AUROC_fixed0.5,precision_fixed0.5,recall_fixed0.5,F1_fixed0.5,MCC_fixed0.5,AUPRC_OOFthr,AUROC_OOFthr,precision_OOFthr,recall_OOFthr,F1_OOFthr,MCC_OOFthr
0,LOCAL_MULTIMODAL,ConcatMLP,TestSet300,0.377218,0.559394,0.432469,0.862458,0.514911,0.397849,0.448873,0.421109,0.432469,0.862458,0.546205,0.355607,0.430764,0.411645
1,POCKET_HYPERGRAPH,SM_PocketHG_Attention,TestSet300,0.397647,0.608889,0.440255,0.867647,0.454417,0.495136,0.473903,0.437803,0.440255,0.867647,0.509835,0.418075,0.459418,0.429625
2,LOCAL_MULTIMODAL,ConcatMLP,TestSetNew46,0.377218,0.559394,0.394729,0.857986,0.455108,0.345882,0.393048,0.358818,0.394729,0.857986,0.473118,0.310588,0.375000,0.347716
3,POCKET_HYPERGRAPH,SM_PocketHG_Attention,TestSetNew46,0.397647,0.608889,0.424158,0.865581,0.451852,0.430588,0.440964,0.401205,0.424158,0.865581,0.487500,0.367059,0.418792,0.386954


HEADLINE:


,family,model,dataset,Dev_CV_AUPRC,Dev_OOF_MCC_threshold,AUPRC_fixed0.5,AUROC_fixed0.5,precision_fixed0.5,recall_fixed0.5,F1_fixed0.5,MCC_fixed0.5,AUPRC_OOFthr,AUROC_OOFthr,precision_OOFthr,recall_OOFthr,F1_OOFthr,MCC_OOFthr
1,POCKET_HYPERGRAPH,SM_PocketHG_Attention,TestSet300,0.397647,0.608889,0.440255,0.867647,0.454417,0.495136,0.473903,0.437803,0.440255,0.867647,0.509835,0.418075,0.459418,0.429625
3,POCKET_HYPERGRAPH,SM_PocketHG_Attention,TestSetNew46,0.397647,0.608889,0.424158,0.865581,0.451852,0.430588,0.440964,0.401205,0.424158,0.865581,0.487500,0.367059,0.418792,0.386954


In [22]:
# ============================================================
# SMALL FINAL EXPERIMENT
# V1 Attention architecture + multi-scale pocket bank
# One-cell: build cache -> Dev CV -> external comparison
# ============================================================
from pathlib import Path
from collections import Counter
import json
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.spatial import cKDTree


SMMS_ROOT = (
    BASE
    / 'results'
    / 'SMALL_V1_ATTENTION_MULTISCALE_POCKETS'
)

SMMS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


SMMS_CACHE = (
    BASE
    / 'cache'
    / 'small_multiscale_pockets_v1'
)

SMMS_CACHE.mkdir(
    parents=True,
    exist_ok=True
)


SMMS_RADII = (
    6.0,
    8.0,
    10.0,
    12.0
)

SMMS_MAX = 64

SMMS_NMS = 0.65

SMMS_FORCE = False


def _jaccard(
    a,
    b
):

    a = set(a)
    b = set(b)

    return (
        len(
            a & b
        )
        /
        max(
            len(
                a | b
            ),
            1
        )
    )


def _build_multiscale_pockets(
    split,
    pid,
    seq
):

    coords = structure_coords(
        split,
        pid,
        seq
    )

    tree = cKDTree(
        coords
    )

    candidates = []


    for seed, aa in enumerate(
        seq
    ):

        if aa not in POCKET_SEEDS:
            continue


        for radius in SMMS_RADII:

            members = tuple(
                sorted(
                    tree.query_ball_point(
                        coords[
                            seed
                        ],
                        radius
                    )
                )
            )


            if len(
                members
            ) < 4:
                continue


            feat22 = (
                pocket_static_features(
                    seq,
                    coords,
                    members
                )
            )


            # Append radius explicitly.
            feat23 = np.concatenate(
                [
                    feat22.astype(
                        np.float32
                    ),

                    np.array(
                        [
                            radius
                            / 12.0
                        ],
                        dtype=np.float32
                    )
                ]
            )


            # Original V1 pocket score
            # + very mild preference against
            # unnecessarily large radius.
            score = (

                1.4
                * feat22[1]

                + 0.8
                * feat22[2]

                + 0.8
                * feat22[3]

                + 0.5
                * feat22[6]

                + 0.5
                * feat22[7]

                - 0.5
                * feat22[5]

                + 0.25
                * feat22[0]

                + 0.15
                * feat22[17]

                - 0.05
                * (
                    radius
                    / 12.0
                )
            )


            candidates.append(
                (
                    float(
                        score
                    ),
                    members,
                    feat23
                )
            )


    # ========================================================
    # Exact duplicate removal
    # ========================================================

    unique = {}


    for (
        score,
        members,
        feat
    ) in candidates:

        if (
            members not in unique
            or
            score
            >
            unique[
                members
            ][0]
        ):

            unique[
                members
            ] = (
                score,
                feat
            )


    ranked = sorted(
        [
            (
                s,
                m,
                f
            )

            for m, (
                s,
                f
            )
            in unique.items()
        ],

        key=lambda x:
            x[0],

        reverse=True
    )


    # ========================================================
    # Jaccard NMS
    # ========================================================

    selected = []


    for item in ranked:

        (
            score,
            members,
            feat
        ) = item


        if any(
            _jaccard(
                members,
                old_members
            )
            >=
            SMMS_NMS

            for _,
            old_members,
            _
            in selected
        ):
            continue


        selected.append(
            item
        )


        if len(
            selected
        ) >= SMMS_MAX:
            break


    edge_res = []
    edge_pocket = []
    pocket_feat = []


    for pidx, (
        _,
        members,
        feat
    ) in enumerate(
        selected
    ):

        pocket_feat.append(
            feat
        )


        for rid in members:

            edge_res.append(
                int(
                    rid
                )
            )

            edge_pocket.append(
                int(
                    pidx
                )
            )


    return (

        np.asarray(
            edge_res,
            dtype=np.int64
        ),

        np.asarray(
            edge_pocket,
            dtype=np.int64
        ),

        np.asarray(
            pocket_feat,
            dtype=np.float32
        ).reshape(
            -1,
            23
        )
    )


# ============================================================
# Build/resume pocket bank
# ============================================================

SMMS_RAM = {}

audit_rows = []


for split, df in (
    SPLIT_DF.items()
):

    od = (
        SMMS_CACHE
        / split
    )

    od.mkdir(
        parents=True,
        exist_ok=True
    )


    for n, r in enumerate(
        df.itertuples(),
        start=1
    ):

        fp = (
            od
            / f'{r.prot_id}.npz'
        )


        good = False


        if (
            fp.exists()
            and
            not SMMS_FORCE
        ):

            try:

                z = np.load(
                    fp
                )


                good = (

                    z[
                        'pocket_feat'
                    ].ndim
                    == 2

                    and

                    z[
                        'pocket_feat'
                    ].shape[1]
                    == 23

                    and

                    len(
                        z[
                            'edge_res'
                        ]
                    )
                    ==
                    len(
                        z[
                            'edge_pocket'
                        ]
                    )
                )


            except Exception:

                good = False


        if not good:

            (
                er,
                ep,
                pf
            ) = (
                _build_multiscale_pockets(
                    split,
                    str(
                        r.prot_id
                    ),
                    r.sequence
                )
            )


            np.savez_compressed(
                fp,
                edge_res=er,
                edge_pocket=ep,
                pocket_feat=pf
            )


        z = np.load(
            fp
        )


        er = z[
            'edge_res'
        ].astype(
            np.int64
        )


        ep = z[
            'edge_pocket'
        ].astype(
            np.int64
        )


        pf = z[
            'pocket_feat'
        ].astype(
            np.float32
        )


        SMMS_RAM[
            (
                split,
                str(
                    r.prot_id
                )
            )
        ] = (
            er,
            ep,
            pf
        )


        covered = (
            len(
                np.unique(
                    er
                )
            )
            if len(er)
            else 0
        )


        audit_rows.append(
            {
                'dataset':
                    split,

                'prot_id':
                    str(
                        r.prot_id
                    ),

                'n_pockets':
                    len(
                        pf
                    ),

                'coverage':
                    covered
                    /
                    max(
                        len(
                            r.sequence
                        ),
                        1
                    ),

                'memberships':
                    len(
                        er
                    ),
            }
        )


        if n % 250 == 0:

            print(
                split,
                n,
                '/',
                len(df)
            )


smms_audit = pd.DataFrame(
    audit_rows
)


print(
    "\nMULTI-SCALE POCKET AUDIT"
)


display(
    smms_audit
    .groupby(
        'dataset'
    )
    .agg(
        mean_pockets=(
            'n_pockets',
            'mean'
        ),

        median_pockets=(
            'n_pockets',
            'median'
        ),

        mean_coverage=(
            'coverage',
            'mean'
        ),

        mean_memberships=(
            'memberships',
            'mean'
        ),
    )
    .reset_index()
)


smms_audit.to_csv(
    SMMS_ROOT
    / 'pocket_audit.csv',
    index=False
)


# ============================================================
# Same V1 Attention architecture.
# Only pocket static dimension 22 -> 23.
# ============================================================

class SmallPocketHypergraphMS(
    SmallPocketHypergraph
):

    def __init__(
        self,
        dims,
        mode='attention',
        latent=192,
        drop=0.2
    ):

        super().__init__(
            dims,
            mode=mode,
            latent=latent,
            moe=False,
            drop=drop
        )


        self.pocket_static = (
            nn.Sequential(
                nn.LayerNorm(
                    23
                ),
                nn.Linear(
                    23,
                    64
                ),
                nn.GELU()
            )
        )


def collate_pocket_ms(
    batch
):

    keys = sorted(
        batch[0][
            'mods'
        ]
    )


    mods = {

        k:
            torch.from_numpy(
                np.concatenate(
                    [
                        b[
                            'mods'
                        ][k]

                        for b
                        in batch
                    ],
                    axis=0
                )
            )
            .float()

        for k
        in keys
    }


    y = torch.from_numpy(
        np.concatenate(
            [
                b[
                    'y'
                ]
                for b
                in batch
            ]
        )
    ).float()


    pf_all = []
    er_all = []
    ep_all = []


    roff = 0
    poff = 0


    for b in batch:

        L = len(
            b[
                'y'
            ]
        )


        P = len(
            b[
                'pocket_feat'
            ]
        )


        if P:

            pf_all.append(
                torch.from_numpy(
                    b[
                        'pocket_feat'
                    ]
                )
                .float()
            )


        if len(
            b[
                'edge_res'
            ]
        ):

            er_all.append(
                torch.from_numpy(
                    b[
                        'edge_res'
                    ]
                )
                .long()
                +
                roff
            )


            ep_all.append(
                torch.from_numpy(
                    b[
                        'edge_pocket'
                    ]
                )
                .long()
                +
                poff
            )


        roff += L
        poff += P


    return {

        'mods':
            mods,

        'y':
            y,

        'edge_res':

            (
                torch.cat(
                    er_all,
                    0
                )

                if er_all

                else torch.zeros(
                    (
                        0,
                    ),
                    dtype=torch.long
                )
            ),

        'edge_pocket':

            (
                torch.cat(
                    ep_all,
                    0
                )

                if ep_all

                else torch.zeros(
                    (
                        0,
                    ),
                    dtype=torch.long
                )
            ),

        'pocket_feat':

            (
                torch.cat(
                    pf_all,
                    0
                )

                if pf_all

                else torch.zeros(
                    (
                        0,
                        23
                    ),
                    dtype=torch.float32
                )
            ),
    }


# ============================================================
# Swap ONLY pocket bank/factory temporarily.
# ============================================================

_OLD_POCKET_RAM = (
    POCKET_RAM
)

_OLD_POCKET_FACTORY = (
    POCKET_FACTORY
)

_OLD_POCKET_STATE_ROOT = (
    POCKET_STATE_ROOT
)

_OLD_COLLATE_POCKET = (
    collate_pocket
)


POCKET_RAM = (
    SMMS_RAM
)


POCKET_FACTORY = {

    'SM_PocketHG_Attention_MS':

        lambda dims:
            SmallPocketHypergraphMS(
                dims,
                mode='attention',
                latent=POCKET_LATENT,
                drop=0.2
            )
}


POCKET_STATE_ROOT = (
    SMMS_ROOT
    / 'states'
)


POCKET_STATE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


collate_pocket = (
    collate_pocket_ms
)


try:

    print(
        "=" * 100
    )

    print(
        "SMALL: V1 ATTENTION "
        "+ MULTI-SCALE POCKET BANK"
    )

    print(
        "=" * 100
    )


    (
        ms_summary,
        ms_folds,
        ms_oof,
        ms_states
    ) = run_pocket_cv(
        'SM_PocketHG_Attention_MS'
    )


    ms_summary_df = pd.DataFrame(
        [
            ms_summary
        ]
    )


    display(
        ms_summary_df
    )


    display(
        ms_folds
    )


    ms_summary_df.to_csv(
        SMMS_ROOT
        / 'dev_cv_summary.csv',
        index=False
    )


    ms_folds.to_csv(
        SMMS_ROOT
        / 'dev_cv_folds.csv',
        index=False
    )


    MS_DEV = float(
        ms_summary[
            'CV_AUPRC_mean'
        ]
    )


    V1_DEV = float(
        pocket_summary.loc[
            pocket_summary
            .architecture
            .eq(
                'SM_PocketHG_Attention'
            ),
            'CV_AUPRC_mean'
        ].iloc[0]
    )


    print(
        f"\nV1 Attention Dev mean AUPRC : "
        f"{V1_DEV:.6f}"
    )


    print(
        f"MS Attention Dev mean AUPRC : "
        f"{MS_DEV:.6f}"
    )


    print(
        f"Delta MS - V1               : "
        f"{MS_DEV - V1_DEV:+.6f}"
    )


    MS_WINS = (
        MS_DEV
        -
        V1_DEV
        >=
        0.002
    )


    print(
        "MS becomes new headline?     :",
        MS_WINS
    )


    # ========================================================
    # Frozen Dev threshold
    # ========================================================

    dev_y = np.concatenate(
        train_df
        .small_binding
        .apply(
            lambda x:
                np.asarray(
                    x,
                    dtype=np.int8
                )
        )
        .tolist()
    )


    (
        ms_thr,
        ms_oof_mcc
    ) = best_mcc_threshold(
        dev_y,
        ms_oof
    )


    # ========================================================
    # External prediction
    # ========================================================

    def _predict_ms_external(
        split
    ):

        ds = PocketProteinDataset(
            split,
            SPLIT_DF[
                split
            ]
        )


        preds = []


        for fold, state in (
            ms_states.items()
        ):

            model = (
                POCKET_FACTORY[
                    'SM_PocketHG_Attention_MS'
                ](
                    DIMS
                )
                .to(
                    DEVICE
                )
            )


            model.load_state_dict(
                state
            )


            p, _ = predict_pocket(
                model,
                ds
            )


            preds.append(
                p
            )


            del model


            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()


        return np.mean(
            np.stack(
                preds
            ),
            axis=0
        )


    ext_rows = []


    for split in [
        'TestSet300',
        'TestSetNew46'
    ]:

        y = np.concatenate(
            SPLIT_DF[
                split
            ]
            .small_binding
            .apply(
                lambda x:
                    np.asarray(
                        x,
                        dtype=np.int8
                    )
            )
            .tolist()
        )


        p = _predict_ms_external(
            split
        )


        m05 = binary_metrics(
            y,
            p,
            0.5
        )


        mdev = binary_metrics(
            y,
            p,
            ms_thr
        )


        row = {

            'dataset':
                split,

            'method':
                'SM_PocketHG_Attention_MS',

            'Dev_CV_AUPRC':
                MS_DEV,

            'Dev_OOF_MCC_threshold':
                ms_thr,
        }


        row.update(
            {
                f'{k}_fixed0.5':
                    v

                for k, v
                in m05.items()
            }
        )


        row.update(
            {
                f'{k}_OOFthr':
                    v

                for k, v
                in mdev.items()
            }
        )


        ext_rows.append(
            row
        )


    ms_external = pd.DataFrame(
        ext_rows
    )


    display(
        ms_external
    )


    ms_external.to_csv(
        SMMS_ROOT
        / 'external_metrics.csv',
        index=False
    )


    # ========================================================
    # Correct comparison:
    # new MS + V1 + bind24 + bind21
    # ========================================================

    cmp_rows = []


    for _, r in (
        ms_external.iterrows()
    ):

        cmp_rows.append(
            {
                'dataset':
                    r[
                        'dataset'
                    ],

                'method':
                    'MS PocketHG Attention',

                'AUPRC':
                    r[
                        'AUPRC_fixed0.5'
                    ],

                'AUROC':
                    r[
                        'AUROC_fixed0.5'
                    ],

                'F1':
                    r[
                        'F1_fixed0.5'
                    ],

                'MCC':
                    r[
                        'MCC_fixed0.5'
                    ],
            }
        )


    # V1 current headline
    for _, r in (
        headline.iterrows()
    ):

        cmp_rows.append(
            {
                'dataset':
                    r[
                        'dataset'
                    ],

                'method':
                    f'V1 {GLOBAL_MODEL}',

                'AUPRC':
                    r[
                        'AUPRC_fixed0.5'
                    ],

                'AUROC':
                    r[
                        'AUROC_fixed0.5'
                    ],

                'F1':
                    r[
                        'F1_fixed0.5'
                    ],

                'MCC':
                    r[
                        'MCC_fixed0.5'
                    ],
            }
        )


    baseline_fp = (
        BASELINE_WORK
        / 'results'
        / 'FULL_DIRECT_BASELINE_METRICS_300_46.csv'
    )


    if baseline_fp.exists():

        bb = pd.read_csv(
            baseline_fp
        )


        if 'label' in bb.columns:

            bb = bb[
                bb.label.eq(
                    'small_binding'
                )
            ].copy()


        if (
            'operating_point'
            in bb.columns
        ):

            bb = bb[
                bb.operating_point.eq(
                    'fixed_0.5'
                )
            ].copy()


        for _, r in (
            bb.iterrows()
        ):

            if str(
                r.method
            ) not in {
                'bindNode24',
                'bindEmbed21DL'
            }:

                continue


            apcol = (
                'AP_AUPRC'
                if (
                    'AP_AUPRC'
                    in bb.columns
                )
                else 'AUPRC'
            )


            cmp_rows.append(
                {
                    'dataset':
                        str(
                            r.dataset
                        ),

                    'method':
                        str(
                            r.method
                        ),

                    'AUPRC':
                        float(
                            r[
                                apcol
                            ]
                        ),

                    'AUROC':
                        float(
                            r.AUROC
                        ),

                    'F1':
                        float(
                            r.global_positive_F1
                        ),

                    'MCC':
                        float(
                            r.global_MCC
                        ),
                }
            )


    ms_compare = (
        pd.DataFrame(
            cmp_rows
        )
        .sort_values(
            [
                'dataset',
                'AUPRC'
            ],
            ascending=[
                True,
                False
            ]
        )
    )


    print(
        "\nFINAL SMALL COMPARISON"
    )


    display(
        ms_compare
    )


    ms_compare.to_csv(
        SMMS_ROOT
        / 'FINAL_COMPARISON.csv',
        index=False
    )


finally:

    POCKET_RAM = (
        _OLD_POCKET_RAM
    )

    POCKET_FACTORY = (
        _OLD_POCKET_FACTORY
    )

    POCKET_STATE_ROOT = (
        _OLD_POCKET_STATE_ROOT
    )

    collate_pocket = (
        _OLD_COLLATE_POCKET
    )


print(
    "\nSaved:",
    SMMS_ROOT
)

DevSet1010 250 / 1010
DevSet1010 500 / 1010
DevSet1010 750 / 1010
DevSet1010 1000 / 1010
TestSet300 250 / 300

MULTI-SCALE POCKET AUDIT


,dataset,mean_pockets,median_pockets,mean_coverage,mean_memberships
0,DevSet1010,52.470297,64.0,0.834215,811.611881
1,TestSet300,57.490000,64.0,0.802372,939.083333
2,TestSetNew46,45.586957,53.0,0.848493,687.173913


SMALL: V1 ATTENTION + MULTI-SCALE POCKET BANK
SM_PocketHG_Attention_MS fold 1 REUSE
SM_PocketHG_Attention_MS fold 2 REUSE
SM_PocketHG_Attention_MS fold 3 REUSE
SM_PocketHG_Attention_MS fold 4 REUSE
SM_PocketHG_Attention_MS fold 5 REUSE


,architecture,CV_AUPRC_mean,CV_AUPRC_std,OOF_AUPRC
0,SM_PocketHG_Attention_MS,0.392769,0.029363,0.372801


,architecture,fold,AUPRC,n_val,n_positive
0,SM_PocketHG_Attention_MS,1,0.402902,32977,2069
1,SM_PocketHG_Attention_MS,2,0.422164,34703,1838
2,SM_PocketHG_Attention_MS,3,0.361797,34034,1629
3,SM_PocketHG_Attention_MS,4,0.361232,33622,1688
4,SM_PocketHG_Attention_MS,5,0.415750,34845,2012



V1 Attention Dev mean AUPRC : 0.397647
MS Attention Dev mean AUPRC : 0.392769
Delta MS - V1               : -0.004878
MS becomes new headline?     : False


,dataset,method,Dev_CV_AUPRC,Dev_OOF_MCC_threshold,AUPRC_fixed0.5,AUROC_fixed0.5,precision_fixed0.5,recall_fixed0.5,F1_fixed0.5,MCC_fixed0.5,AUPRC_OOFthr,AUROC_OOFthr,precision_OOFthr,recall_OOFthr,F1_OOFthr,MCC_OOFthr
0,TestSet300,SM_PocketHG_Attention_MS,0.392769,0.465354,0.441991,0.863503,0.511378,0.425755,0.464655,0.434571,0.441991,0.863503,0.489739,0.452125,0.470181,0.436865
1,TestSetNew46,SM_PocketHG_Attention_MS,0.392769,0.465354,0.419632,0.857929,0.452381,0.357647,0.399474,0.363788,0.419632,0.857929,0.453039,0.385882,0.416773,0.379085



FINAL SMALL COMPARISON


,dataset,method,AUPRC,AUROC,F1,MCC
0,TestSet300,MS PocketHG Attention,0.441991,0.863503,0.464655,0.434571
2,TestSet300,V1 SM_PocketHG_Attention,0.440255,0.867647,0.473903,0.437803
6,TestSet300,bindNode24,0.431709,0.869662,0.468245,0.432216
4,TestSet300,bindEmbed21DL,0.394041,0.864302,0.435777,0.396702
3,TestSetNew46,V1 SM_PocketHG_Attention,0.424158,0.865581,0.440964,0.401205
1,TestSetNew46,MS PocketHG Attention,0.419632,0.857929,0.399474,0.363788
7,TestSetNew46,bindNode24,0.380681,0.872435,0.407911,0.367430
5,TestSetNew46,bindEmbed21DL,0.315583,0.857311,0.374857,0.327813



Saved: /content/drive/MyDrive/Protein-binding-Phi/results/SMALL_V1_ATTENTION_MULTISCALE_POCKETS


# Part 2 — V3: re-evaluate the final model, improve it, bootstrap everything


In [23]:
# ============================================================
# V3-0. Task configuration (names of the original notebook's objects)
# ============================================================
V3 = {
    "TASK": "SMALL",
    "LABEL": "small_binding",
    "V1": "SM_PocketHG_Attention",
    "MS": "SM_PocketHG_Attention_MS",
    "RAM": "POCKET_RAM",
    "MS_RAM": "SMMS_RAM",
    "FACTORY": "POCKET_FACTORY",
    "STATE_ROOT": "POCKET_STATE_ROOT",
    "COLLATE": "collate_pocket",
    "MS_COLLATE": "collate_pocket_ms",
    "DATASET": "PocketProteinDataset",
    "PREDICT": "predict_pocket",
    "RUN_CV": "run_pocket_cv",
    "MS_CLASS": "SmallPocketHypergraphMS",
    "LATENT": "POCKET_LATENT",
    "V1_OOF": "POCKET_OOF",
    "V1_STATES": "POCKET_STATES",
    "MS_OOF": "ms_oof",
    "MS_STATES": "ms_states",
    "ORIG_W_V1": 0.6,
    "ORIG_DEV": 0.4092833208152622,
    "ORIG_V1_DEV": 0.39764655317751607,
    "ORIG_EXT": {
        "TestSet300": 0.446015,
        "TestSetNew46": 0.427383
    }
}


In [24]:
# ============================================================
# V3-1. Setup: helpers, bootstrap (B = 10000), member store
#       Everything below runs AFTER the original cells above (all of them REUSE their Drive caches/states).
# ============================================================
import re, time, hashlib
from contextlib import contextmanager
from sklearn.metrics import average_precision_score, roc_auc_score

G = globals()
LABEL = V3['LABEL']
SPLITS_EXT = ('TestSet300', 'TestSetNew46')
B_BOOT, BOOT_SEED = 10000, 2026
MARGIN = 0.002   # complexity margin of the original notebooks
V3_ROOT = BASE/'results'/f"{V3['TASK']}_V3_IMPROVE"
PRED_DIR = V3_ROOT/'member_preds'
PRED_DIR.mkdir(parents=True, exist_ok=True)

# originals captured once (the cells above restored them after the MS run)
ORIG_FACTORY = dict(G[V3['FACTORY']])
ORIG_SPEC = {k: list(v) for k, v in MODALITY_SPEC.items()}
ORIG_SEED = SEED
ORIG_REP_NAME = BEST_REP_NAME
PID0 = str(train_df.iloc[0].prot_id)

def _labels(split):
    return np.concatenate(SPLIT_DF[split][LABEL].apply(lambda x: np.asarray(x, np.int8)).tolist()).astype(np.int64)

Y = {s: _labels(s) for s in SPLIT_DF}                       # DevSet1010 in train_df (= OOF) order
PROT = {s: np.repeat(np.arange(len(df)), df.sequence.str.len().to_numpy()) for s, df in SPLIT_DF.items()}
_off, _cur = {}, 0
for r in train_df.itertuples():
    _off[str(r.prot_id)] = (_cur, _cur + len(r.sequence)); _cur += len(r.sequence)
FOLD_IDX = {f: np.concatenate([np.arange(*_off[str(p)]) for p in ids]) for f, ids in OFFICIAL_FOLDS.items()}

def fold_aps(p):
    return np.array([average_precision_score(Y['DevSet1010'][i], p[i]) for i in FOLD_IDX.values()])

def fold_mean(p):
    return float(fold_aps(p).mean())

@contextmanager
def swapped(**kv):
    old = {k: G[k] for k in kv}
    G.update(kv)
    try:
        yield
    finally:
        G.update(old)

def env_for(ms=False, extra=(), seed=None, name=None):
    """Globals of the original trainer for one variant: modalities (+extra), seed, V1 or MS patch/pocket bank."""
    spec = {**{k: list(v) for k, v in ORIG_SPEC.items()}, **{m: [m] for m in extra}}
    dims = {m: concat_blocks(ctx.stores['train'], PID0, b).shape[1] for m, b in spec.items()}
    arch = V3['MS'] if ms else V3['V1']
    if ms:
        ms_cls = G[V3['MS_CLASS']]
        factory = {arch: (lambda d: ms_cls(d, mode='attention', latent=G[V3['LATENT']], drop=0.2))}
    else:
        factory = {arch: ORIG_FACTORY[arch]}
    env = {'MODALITY_SPEC': spec, 'DIMS': dims, V3['FACTORY']: factory,
           'SEED': ORIG_SEED if seed is None else seed,
           'BEST_REP_NAME': ORIG_REP_NAME + ''.join(f'+{m}' for m in extra)}
    if ms:
        env.update({V3['RAM']: G[V3['MS_RAM']], V3['COLLATE']: G[V3['MS_COLLATE']]})
    if name is not None:
        env[V3['STATE_ROOT']] = V3_ROOT/'states'/name
    return arch, env

def predict_split(arch, states, split):
    ds = G[V3['DATASET']](split, SPLIT_DF[split])
    preds = []
    for fold in sorted(states):
        model = G[V3['FACTORY']][arch](G['DIMS']).to(DEVICE)
        model.load_state_dict(states[fold])
        p, _ = G[V3['PREDICT']](model, ds)
        preds.append(np.asarray(p, np.float64))
        del model
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
    return np.mean(np.stack(preds), axis=0)

MEMBERS = {}

def _save_member(name, oof, ext, info):
    np.savez(PRED_DIR/f'{name}.npz', oof=oof, **ext, oof_sha=hashlib.sha1(np.ascontiguousarray(oof)).hexdigest())
    MEMBERS[name] = {'oof': oof, **ext, **info}

def _cached_ext(name, oof):
    fp = PRED_DIR/f'{name}.npz'
    if fp.exists():
        z = np.load(fp, allow_pickle=False)
        if str(z['oof_sha']) == hashlib.sha1(np.ascontiguousarray(oof)).hexdigest() and all(s in z.files for s in SPLITS_EXT):
            return {s: z[s] for s in SPLITS_EXT}
    return None

def run_member(name, ms=False, extra=(), seed=None):
    """Official 5-fold Dev CV with the ORIGINAL trainer (per-fold resume from Drive), then Test300/46 = mean of folds."""
    arch, env = env_for(ms, extra, seed, name)
    with swapped(**env):
        summ, folds, oof, states = G[V3['RUN_CV']](arch)
        oof = np.asarray(oof, np.float64)
        ext = _cached_ext(name, oof) or {s: predict_split(arch, states, s) for s in SPLITS_EXT}
    _save_member(name, oof, ext, {'ms': ms, 'extra': tuple(extra), 'seed': env['SEED']})
    print(f'{name}: Dev fold-mean AUPRC {fold_mean(oof):.6f} | Test300 {average_precision_score(Y["TestSet300"], ext["TestSet300"]):.4f} '
          f'| Test46 {average_precision_score(Y["TestSetNew46"], ext["TestSetNew46"]):.4f}')
    return MEMBERS[name]

# ---------- bootstrap (fast weighted metrics == sklearn with sample_weight; checked below) ----------
def _curve(y, w, order, last):
    ys, ws = y[order], w[order]
    return np.cumsum(ws * ys)[last], np.cumsum(ws * (1 - ys))[last]

def prep(p):
    order = np.argsort(-p, kind='mergesort')
    ps = p[order]
    return order, np.r_[np.flatnonzero(np.diff(ps) != 0), len(ps) - 1]

def w_auprc(y, w, pr):
    tp, fp = _curve(y, w, *pr)
    return float(np.sum(np.diff(np.r_[0.0, tp / tp[-1]]) * tp / np.maximum(tp + fp, 1e-300)))

def w_auroc(y, w, pr):
    tp, fp = _curve(y, w, *pr)
    tpr, fpr = np.r_[0.0, tp / tp[-1]], np.r_[0.0, fp / fp[-1]]
    return float(np.sum(np.diff(fpr) * (tpr[1:] + tpr[:-1]) / 2))

def w_f1_mcc(y, w, pred):
    tp = np.sum(w * y * pred); fp = np.sum(w * (1 - y) * pred)
    fn = np.sum(w * y * (1 - pred)); tn = np.sum(w * (1 - y) * (1 - pred))
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return float(2 * tp / max(2 * tp + fp + fn, 1e-300)), float((tp * tn - fp * fn) / den) if den > 0 else 0.0

def _metrics(y, w, pr, pred):
    f1, mcc = w_f1_mcc(y, w, pred)
    return {'AUPRC': w_auprc(y, w, pr), 'AUROC': w_auroc(y, w, pr), 'F1@DevThr': f1, 'MCC@DevThr': mcc}

def paired_bootstrap(y, p_ref, p_new, thr_ref, thr_new, prot, B=B_BOOT, seed=BOOT_SEED):
    """Resample proteins with replacement (same proteins for both models)."""
    y = y.astype(float)
    n = int(prot.max()) + 1
    pr_r, pr_n = prep(p_ref), prep(p_new)
    pd_r, pd_n = (p_ref >= thr_ref).astype(float), (p_new >= thr_new).astype(float)
    one = np.ones_like(y)
    pt_r, pt_n = _metrics(y, one, pr_r, pd_r), _metrics(y, one, pr_n, pd_n)
    rng = np.random.default_rng(seed)
    R, N = {k: [] for k in pt_r}, {k: [] for k in pt_r}
    for _ in range(B):
        w = np.bincount(rng.integers(0, n, n), minlength=n)[prot].astype(float)
        if np.sum(w * y) == 0:
            continue
        mr, mn = _metrics(y, w, pr_r, pd_r), _metrics(y, w, pr_n, pd_n)
        for k in pt_r:
            R[k].append(mr[k]); N[k].append(mn[k])
    rows = []
    for k in pt_r:
        r, nw = np.asarray(R[k]), np.asarray(N[k])
        d = nw - r
        rows.append({'metric': k, 'ref': pt_r[k], 'ref_CI95': f'[{np.percentile(r, 2.5):.4f}, {np.percentile(r, 97.5):.4f}]',
                     'new': pt_n[k], 'new_CI95': f'[{np.percentile(nw, 2.5):.4f}, {np.percentile(nw, 97.5):.4f}]',
                     'delta': pt_n[k] - pt_r[k], 'delta_CI95_lo': np.percentile(d, 2.5), 'delta_CI95_hi': np.percentile(d, 97.5),
                     'P(new>ref)': float(np.mean(d > 0)), 'n_boot': len(d)})
    return pd.DataFrame(rows)

def _pooled(a, b):
    return np.concatenate([a, b])

def compare(ref_name, ref, new_name, new, sets=('DevSet1010', 'TestSet300', 'TestSetNew46', 'POOLED_TEST')):
    """ref/new: dict with oof, TestSet300, TestSetNew46, thr (Dev-OOF MCC threshold). Saves + returns the table."""
    out = []
    for s in sets:
        if s == 'DevSet1010':
            y, pr, pn, prot = Y[s], ref['oof'], new['oof'], PROT[s]
        elif s == 'POOLED_TEST':
            y = _pooled(Y['TestSet300'], Y['TestSetNew46'])
            pr, pn = _pooled(ref['TestSet300'], ref['TestSetNew46']), _pooled(new['TestSet300'], new['TestSetNew46'])
            prot = _pooled(PROT['TestSet300'], PROT['TestSetNew46'] + PROT['TestSet300'].max() + 1)
        else:
            y, pr, pn, prot = Y[s], ref[s], new[s], PROT[s]
        t0 = time.time()
        r = paired_bootstrap(y, pr, pn, ref['thr'], new['thr'], prot)
        r.insert(0, 'dataset', s)
        r.insert(0, 'new_model', new_name)
        r.insert(0, 'ref_model', ref_name)
        out.append(r)
        print(f'  bootstrap {new_name} vs {ref_name} | {s}: {time.time() - t0:.0f}s')
    out = pd.concat(out, ignore_index=True)
    out.to_csv(V3_ROOT/f'BOOT_{new_name}_vs_{ref_name}.csv', index=False)
    return out

def show_auprc(tab):
    t = tab[tab.metric.eq('AUPRC')][['dataset', 'ref', 'new', 'delta', 'delta_CI95_lo', 'delta_CI95_hi', 'P(new>ref)']]
    display(t.round(4))
    return t

# the fast metrics must equal sklearn (with and without bootstrap weights)
_y, _p = Y['TestSet300'].astype(float), np.random.default_rng(0).random(len(Y['TestSet300']))
_w = np.bincount(np.random.default_rng(1).integers(0, 300, 300), minlength=300)[PROT['TestSet300']].astype(float)
for _ww in (np.ones_like(_y), _w):
    assert abs(w_auprc(_y, _ww, prep(_p)) - average_precision_score(_y, _p, sample_weight=_ww)) < 1e-9
    assert abs(w_auroc(_y, _ww, prep(_p)) - roc_auc_score(_y, _p, sample_weight=_ww)) < 1e-9
print('V3 setup: PASS | task', V3['TASK'], '| original modalities', ORIG_SPEC, '| seed', ORIG_SEED)


V3 setup: PASS | task SMALL | original modalities {'ESM': ['ESM'], 'POCKET': ['ACCESS', 'SM_POCKET3D44', 'SM_TOPO16']} | seed 2026


In [25]:
# ============================================================
# V3-2. Current final model (C0) = the original weights, re-evaluated + bootstrap CI
#       V1 / MS OOF come from the saved fold predictions -> Dev is exactly the original.
# ============================================================
w0 = float(np.linspace(0.0, 1.0, 21)[int(round(V3['ORIG_W_V1'] * 20))])   # exact grid value of the original search
v1_oof = np.asarray(G[V3['V1_OOF']][V3['V1']], np.float64)
ms_oof_ = np.asarray(G[V3['MS_OOF']], np.float64)

arch, env = env_for(ms=False)
with swapped(**env):
    v1_ext = _cached_ext('V1_s0', v1_oof) or {s: predict_split(arch, G[V3['V1_STATES']][V3['V1']], s) for s in SPLITS_EXT}
_save_member('V1_s0', v1_oof, v1_ext, {'ms': False, 'extra': (), 'seed': ORIG_SEED})
arch, env = env_for(ms=True)
with swapped(**env):
    ms_ext = _cached_ext('MS_s0', ms_oof_) or {s: predict_split(arch, G[V3['MS_STATES']], s) for s in SPLITS_EXT}
_save_member('MS_s0', ms_oof_, ms_ext, {'ms': True, 'extra': (), 'seed': ORIG_SEED})

def combo(names, weights):
    out = {k: sum(w * MEMBERS[n][k] for n, w in zip(names, weights)) for k in ('oof', *SPLITS_EXT)}
    out['thr'] = best_mcc_threshold(Y['DevSet1010'], out['oof'])[0]   # Dev-OOF MCC threshold (original convention)
    out['members'], out['weights'] = list(names), [float(w) for w in weights]
    return out

CANDS = {'C0_current': combo(['V1_s0', 'MS_s0'], [w0, 1 - w0])}
C0 = CANDS['C0_current']
print(f"C0 Dev fold-mean AUPRC {fold_mean(C0['oof']):.10f} (original {V3['ORIG_DEV']:.10f})")
assert abs(fold_mean(C0['oof']) - V3['ORIG_DEV']) < 1e-9, 'Dev of the current final is not the original one'
assert abs(fold_mean(v1_oof) - V3['ORIG_V1_DEV']) < 1e-9
for s in SPLITS_EXT:
    a = average_precision_score(Y[s], C0[s])
    print(f"C0 {s}: AUPRC {a:.6f} (original {V3['ORIG_EXT'][s]:.6f}, diff {a - V3['ORIG_EXT'][s]:+.1e})")

# CI of the current final alone (bootstrap of C0 against itself gives its own CI columns)
C0_CI = compare('C0_current', C0, 'C0_current', C0)
display(C0_CI[['dataset', 'metric', 'ref', 'ref_CI95']].round(4))


C0 Dev fold-mean AUPRC 0.4092833208 (original 0.4092833208)
C0 TestSet300: AUPRC 0.446015 (original 0.446015, diff -1.1e-07)
C0 TestSetNew46: AUPRC 0.427383 (original 0.427383, diff +2.3e-07)
  bootstrap C0_current vs C0_current | DevSet1010: 209s
  bootstrap C0_current vs C0_current | TestSet300: 76s
  bootstrap C0_current vs C0_current | TestSetNew46: 8s
  bootstrap C0_current vs C0_current | POOLED_TEST: 84s


,dataset,metric,ref,ref_CI95
0,DevSet1010,AUPRC,0.3950,"[0.3646, 0.4247]"
1,DevSet1010,AUROC,0.8585,"[0.8479, 0.8688]"
2,DevSet1010,F1@DevThr,0.4268,"[0.4049, 0.4481]"
3,DevSet1010,MCC@DevThr,0.3930,"[0.3709, 0.4146]"
4,TestSet300,AUPRC,0.4460,"[0.4041, 0.4875]"
5,TestSet300,AUROC,0.8685,"[0.8490, 0.8872]"
6,TestSet300,F1@DevThr,0.4709,"[0.4373, 0.5021]"
7,TestSet300,MCC@DevThr,0.4372,"[0.4030, 0.4700]"
8,TestSetNew46,AUPRC,0.4274,"[0.3026, 0.5551]"
9,TestSetNew46,AUROC,0.8657,"[0.8149, 0.9181]"


In [26]:
# ============================================================
# V3-3. New per-residue features (cached on Drive, attached to the stores under NEW block names)
#   ESMIF_EF : ESM-IF1 encoder (512-d) on the same ESMFold chain-A structures (computed by the metal V4-FIX run)
#   PROTT5   : ProtT5-XL-U50 half-precision encoder (1024-d), the embedding behind bindEmbed21DL / bindNode24
# ============================================================
ESMIF_ROOT = BASE/'cache'/'gear_metal_v3'/'esmif_esmfold_v1'
PROTT5_ROOT = BASE/'cache'/'prott5_xl_half_v1'
PROTT5_MODEL = 'Rostlab/prot_t5_xl_half_uniref50-enc'

def attach(split, pid, block, arr):
    rec = ctx.stores[STORE_KEY[split]][str(pid)]
    tgt = rec['features'] if isinstance(rec, dict) and isinstance(rec.get('features'), dict) else rec
    tgt[block] = arr   # kept float16 (concat_blocks casts per protein) -> half the RAM

def _ok(fp, L, d):
    try:
        a = np.load(fp, mmap_mode='r')
        return a.shape == (L, d) and bool(np.isfinite(a).all())
    except Exception:
        return False

# --- ESM-IF
n_esmif = 0
for split, df in SPLIT_DF.items():
    for r in df.itertuples():
        fp = ESMIF_ROOT/split/'esm_if'/f'{r.prot_id}.npy'
        assert _ok(fp, len(r.sequence), 512), f'missing ESM-IF {split}/{r.prot_id} (run the metal V4 notebook first)'
        attach(split, r.prot_id, 'ESMIF_EF', np.load(fp).astype(np.float16))
        n_esmif += 1
print('ESMIF_EF attached:', n_esmif)

# --- ProtT5 (computed once, ~10 min on a T4)
todo = [(s, str(r.prot_id), r.sequence) for s, df in SPLIT_DF.items() for r in df.itertuples()
        if not _ok(PROTT5_ROOT/s/f'{r.prot_id}.npy', len(r.sequence), 1024)]
print('ProtT5 to compute:', len(todo))
if todo:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentencepiece', 'protobuf'], check=True)
    from transformers import T5EncoderModel
    try:   # official ProtT5 usage: space-separated residues, one token per residue + </s>
        from transformers import T5Tokenizer
        _tok = T5Tokenizer.from_pretrained(PROTT5_MODEL, do_lower_case=False)
        def t5_ids(clean):
            ids = _tok(' '.join(clean), add_special_tokens=True)['input_ids']
            if len(ids) != len(clean) + 1:   # residues merged -> map residue pieces explicitly
                ids = _tok.convert_tokens_to_ids([f'\u2581{a}' for a in clean]) + [_tok.eos_token_id]
            return ids, _tok.unk_token_id
    except Exception as e:   # transformers without a usable T5 tokenizer -> the same SentencePiece model directly
        print('T5Tokenizer unavailable -> SentencePiece:', repr(e))
        import sentencepiece as spm
        from huggingface_hub import hf_hub_download
        _sp = spm.SentencePieceProcessor(model_file=hf_hub_download(PROTT5_MODEL, 'spiece.model'))
        def t5_ids(clean):
            return [_sp.piece_to_id(f'\u2581{a}') for a in clean] + [_sp.eos_id()], _sp.unk_id()
    t5 = T5EncoderModel.from_pretrained(PROTT5_MODEL).to(DEVICE).eval()
    t5 = t5.half() if DEVICE.type == 'cuda' else t5.float()   # Rostlab usage: half precision on GPU
    for n, (s, pid, seq) in enumerate(sorted(todo, key=lambda x: len(x[2])), start=1):
        clean = re.sub(r'[UZOB]', 'X', seq)
        ids, unk = t5_ids(clean)
        assert len(ids) == len(seq) + 1 and unk not in ids[:-1], (pid, len(ids), len(seq))
        ids = torch.tensor(ids, dtype=torch.long)
        with torch.no_grad():
            h = t5(input_ids=ids[None].to(DEVICE), attention_mask=torch.ones_like(ids)[None].to(DEVICE)).last_hidden_state
        arr = h[0, :len(seq)].float().cpu().numpy().astype(np.float16)
        assert np.isfinite(arr).all(), pid
        fp = PROTT5_ROOT/s/f'{pid}.npy'
        fp.parent.mkdir(parents=True, exist_ok=True)
        tmp = fp.with_name(f'.tmp.{fp.name}')
        np.save(tmp, arr)
        tmp.replace(fp)
        if n % 200 == 0:
            print('ProtT5', n, '/', len(todo))
    del t5
    torch.cuda.empty_cache()
for split, df in SPLIT_DF.items():
    for r in df.itertuples():
        attach(split, r.prot_id, 'PROTT5', np.load(PROTT5_ROOT/split/f'{r.prot_id}.npy'))
print('PROTT5 attached:', sum(len(df) for df in SPLIT_DF.values()))


ESMIF_EF attached: 1356
ProtT5 to compute: 1356


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/238k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/656 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.42GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.42GB            

model.safetensors: downloading bytes:           |  0.00B            

ProtT5 200 / 1356
ProtT5 400 / 1356
ProtT5 600 / 1356
ProtT5 800 / 1356
ProtT5 1000 / 1356
ProtT5 1200 / 1356
PROTT5 attached: 1356


In [27]:
# ============================================================
# V3-4. Phase A — new modality on the ORIGINAL V1 hypergraph (same trainer, same seed), Dev-only decision
#       Each variant: paired fold test vs V1 + protein bootstrap (B = 10000) vs V1.
# ============================================================
from scipy.stats import ttest_rel

PHASE_A = [('ESMIF_EF',), ('PROTT5',), ('ESMIF_EF', 'PROTT5')]
suffix = lambda extra: ''.join(f'+{m}' for m in extra)
V1 = MEMBERS['V1_s0']
V1['thr'] = best_mcc_threshold(Y['DevSet1010'], V1['oof'])[0]
rows, BOOT_A = [], []
for extra in PHASE_A:
    name = f'V1{suffix(extra)}_s0'
    m = run_member(name, extra=extra)
    m['thr'] = best_mcc_threshold(Y['DevSet1010'], m['oof'])[0]
    fa, fb = fold_aps(V1['oof']), fold_aps(m['oof'])
    rows.append({'variant': name, 'extra': extra, 'Dev_fold_mean': fb.mean(), 'delta_vs_V1': fb.mean() - fa.mean(),
                 'folds_up': int((fb > fa).sum()), 'paired_t_p': float(ttest_rel(fb, fa).pvalue)})
    tab = compare('V1_s0', V1, name, m)
    BOOT_A.append(tab)
    show_auprc(tab)
phase_a = pd.DataFrame(rows).sort_values('Dev_fold_mean', ascending=False).reset_index(drop=True)
phase_a.to_csv(V3_ROOT/'PHASE_A_modalities.csv', index=False)
display(phase_a.round(4))
best = phase_a.iloc[0]
BEST_MOD = tuple(best.extra) if best.delta_vs_V1 >= MARGIN else ()
print('PHASE A decision (Dev only):', f'{best.variant} (+{best.delta_vs_V1:.4f})' if BEST_MOD else 'no modality beats V1 by the margin')


SM_PocketHG_Attention fold 1 TRAIN
SM_PocketHG_Attention fold 1 epoch 1 loss 0.4266 val_AP 0.4627 best 0.4627 alpha 0.0498
SM_PocketHG_Attention fold 1 epoch 5 loss 0.223 val_AP 0.46593 best 0.4842 alpha 0.0494
SM_PocketHG_Attention fold 2 TRAIN
SM_PocketHG_Attention fold 2 epoch 1 loss 0.4404 val_AP 0.43068 best 0.43068 alpha 0.0499
SM_PocketHG_Attention fold 2 epoch 5 loss 0.2296 val_AP 0.47218 best 0.47218 alpha 0.0493
SM_PocketHG_Attention fold 2 epoch 10 loss 0.0559 val_AP 0.43137 best 0.47689 alpha 0.0493
SM_PocketHG_Attention fold 3 TRAIN
SM_PocketHG_Attention fold 3 epoch 1 loss 0.4393 val_AP 0.42953 best 0.42953 alpha 0.0498
SM_PocketHG_Attention fold 3 epoch 5 loss 0.2246 val_AP 0.42523 best 0.45493 alpha 0.0495
SM_PocketHG_Attention fold 4 TRAIN
SM_PocketHG_Attention fold 4 epoch 1 loss 0.4399 val_AP 0.38409 best 0.38409 alpha 0.0498
SM_PocketHG_Attention fold 4 epoch 5 loss 0.2134 val_AP 0.40995 best 0.4168 alpha 0.0493
SM_PocketHG_Attention fold 5 TRAIN
SM_PocketHG_Attenti

,dataset,ref,new,delta,delta_CI95_lo,delta_CI95_hi,P(new>ref)
0,DevSet1010,0.3763,0.4471,0.0708,0.0537,0.0873,1.0000
4,TestSet300,0.4403,0.4865,0.0462,0.0296,0.0626,1.0000
8,TestSetNew46,0.4242,0.4697,0.0455,0.0020,0.0852,0.9801
12,POOLED_TEST,0.4381,0.4849,0.0467,0.0307,0.0618,1.0000


SM_PocketHG_Attention fold 1 TRAIN
SM_PocketHG_Attention fold 1 epoch 1 loss 0.4468 val_AP 0.36143 best 0.36143 alpha 0.0498
SM_PocketHG_Attention fold 1 epoch 5 loss 0.2049 val_AP 0.40034 best 0.40878 alpha 0.0493
SM_PocketHG_Attention fold 2 TRAIN
SM_PocketHG_Attention fold 2 epoch 1 loss 0.4491 val_AP 0.36581 best 0.36581 alpha 0.0498
SM_PocketHG_Attention fold 2 epoch 5 loss 0.214 val_AP 0.44623 best 0.44623 alpha 0.0496
SM_PocketHG_Attention fold 2 epoch 10 loss 0.0484 val_AP 0.44006 best 0.44623 alpha 0.0494
SM_PocketHG_Attention fold 3 TRAIN
SM_PocketHG_Attention fold 3 epoch 1 loss 0.451 val_AP 0.33167 best 0.33167 alpha 0.0498
SM_PocketHG_Attention fold 3 epoch 5 loss 0.2012 val_AP 0.3779 best 0.38876 alpha 0.0494
SM_PocketHG_Attention fold 3 epoch 10 loss 0.0415 val_AP 0.35347 best 0.38876 alpha 0.0491
SM_PocketHG_Attention fold 4 TRAIN
SM_PocketHG_Attention fold 4 epoch 1 loss 0.4382 val_AP 0.32904 best 0.32904 alpha 0.0499
SM_PocketHG_Attention fold 4 epoch 5 loss 0.2077 va

,dataset,ref,new,delta,delta_CI95_lo,delta_CI95_hi,P(new>ref)
0,DevSet1010,0.3763,0.4033,0.0270,0.0137,0.0402,0.9999
4,TestSet300,0.4403,0.4577,0.0175,0.0063,0.0284,0.9988
8,TestSetNew46,0.4242,0.4104,-0.0138,-0.0347,0.0085,0.1116
12,POOLED_TEST,0.4381,0.4527,0.0146,0.0046,0.0248,0.9978


SM_PocketHG_Attention fold 1 TRAIN
SM_PocketHG_Attention fold 1 epoch 1 loss 0.4285 val_AP 0.46569 best 0.46569 alpha 0.0498
SM_PocketHG_Attention fold 1 epoch 5 loss 0.1713 val_AP 0.46518 best 0.4931 alpha 0.0493
SM_PocketHG_Attention fold 2 TRAIN
SM_PocketHG_Attention fold 2 epoch 1 loss 0.4241 val_AP 0.43087 best 0.43087 alpha 0.0497
SM_PocketHG_Attention fold 2 epoch 5 loss 0.1815 val_AP 0.46952 best 0.47483 alpha 0.0493
SM_PocketHG_Attention fold 3 TRAIN
SM_PocketHG_Attention fold 3 epoch 1 loss 0.4223 val_AP 0.44065 best 0.44065 alpha 0.0499
SM_PocketHG_Attention fold 3 epoch 5 loss 0.1783 val_AP 0.43375 best 0.45633 alpha 0.0492
SM_PocketHG_Attention fold 4 TRAIN
SM_PocketHG_Attention fold 4 epoch 1 loss 0.4331 val_AP 0.40274 best 0.40274 alpha 0.0498
SM_PocketHG_Attention fold 4 epoch 5 loss 0.1907 val_AP 0.42273 best 0.43002 alpha 0.0487
SM_PocketHG_Attention fold 5 TRAIN
SM_PocketHG_Attention fold 5 epoch 1 loss 0.4231 val_AP 0.42729 best 0.42729 alpha 0.0497
SM_PocketHG_Atte

,dataset,ref,new,delta,delta_CI95_lo,delta_CI95_hi,P(new>ref)
0,DevSet1010,0.3763,0.4529,0.0765,0.0567,0.0955,1.0000
4,TestSet300,0.4403,0.4889,0.0486,0.0310,0.0664,1.0000
8,TestSetNew46,0.4242,0.4571,0.0330,-0.0083,0.0780,0.9456
12,POOLED_TEST,0.4381,0.4861,0.0480,0.0306,0.0644,1.0000


,variant,extra,Dev_fold_mean,delta_vs_V1,folds_up,paired_t_p
0,V1+ESMIF_EF+PROTT5_s0,"(ESMIF_EF, PROTT5)",0.4645,0.0669,5,0.0098
1,V1+ESMIF_EF_s0,"(ESMIF_EF,)",0.4602,0.0625,5,0.0089
2,V1+PROTT5_s0,"(PROTT5,)",0.4106,0.0129,4,0.1409


PHASE A decision (Dev only): V1+ESMIF_EF+PROTT5_s0 (+0.0669)


In [28]:
# ============================================================
# V3-5. Phase B — MS hypergraph with the chosen modality; Phase C — 2 extra seeds for the final pair
#       (seed bagging: mean of 3 independently trained 5-fold models per member)
# ============================================================
PAIR = ('V1' + suffix(BEST_MOD), 'MS' + suffix(BEST_MOD))   # V1<mods>_s0 already trained in Phase A
if BEST_MOD:
    run_member(f'{PAIR[1]}_s0', ms=True, extra=BEST_MOD)
EXTRA_SEEDS = (ORIG_SEED + 1000, ORIG_SEED + 2000)
for k, sd in enumerate(EXTRA_SEEDS, start=1):
    run_member(f'{PAIR[0]}_s{k}', ms=False, extra=BEST_MOD, seed=sd)
    run_member(f'{PAIR[1]}_s{k}', ms=True, extra=BEST_MOD, seed=sd)
for base in PAIR:
    names = [f'{base}_s{k}' for k in range(len(EXTRA_SEEDS) + 1)]
    bag = {k: np.mean([MEMBERS[n][k] for n in names], axis=0) for k in ('oof', *SPLITS_EXT)}
    MEMBERS[f'{base}_bag3'] = bag
    print(f'{base}: seeds {[round(fold_mean(MEMBERS[n]["oof"]), 4) for n in names]} -> bag3 {fold_mean(bag["oof"]):.4f}')


SM_PocketHG_Attention_MS fold 1 TRAIN
SM_PocketHG_Attention_MS fold 1 epoch 1 loss 0.4252 val_AP 0.47542 best 0.47542 alpha 0.0498
SM_PocketHG_Attention_MS fold 1 epoch 5 loss 0.1805 val_AP 0.46428 best 0.49375 alpha 0.0491
SM_PocketHG_Attention_MS fold 2 TRAIN
SM_PocketHG_Attention_MS fold 2 epoch 1 loss 0.4286 val_AP 0.44657 best 0.44657 alpha 0.0498
SM_PocketHG_Attention_MS fold 2 epoch 5 loss 0.1814 val_AP 0.467 best 0.49198 alpha 0.0492
SM_PocketHG_Attention_MS fold 3 TRAIN
SM_PocketHG_Attention_MS fold 3 epoch 1 loss 0.428 val_AP 0.42408 best 0.42408 alpha 0.0498
SM_PocketHG_Attention_MS fold 3 epoch 5 loss 0.1844 val_AP 0.42678 best 0.45042 alpha 0.0487
SM_PocketHG_Attention_MS fold 4 TRAIN
SM_PocketHG_Attention_MS fold 4 epoch 1 loss 0.4212 val_AP 0.38001 best 0.38001 alpha 0.0499
SM_PocketHG_Attention_MS fold 4 epoch 5 loss 0.1956 val_AP 0.4224 best 0.42769 alpha 0.0488
SM_PocketHG_Attention_MS fold 4 epoch 10 loss 0.0395 val_AP 0.41053 best 0.42769 alpha 0.0489
SM_PocketHG_At

In [29]:
# ============================================================
# V3-6. Candidate final models, Dev-only decision, then external + bootstrap (B = 10000) vs the current final
#   weights: grid on Dev (step 0.05 for pairs, 0.1 for 4 members) maximising the official 5-fold mean AUPRC
#   decision score: CROSS-FITTED Dev (weights chosen on 4 folds, scored on the 5th) -> no optimism from the search
#   rule (fixed now): simplest first; a candidate replaces the winner only if cross-fitted gain >= 0.002
# ============================================================
from itertools import product

def simplex(k, step):
    n = int(round(1 / step))
    return [np.array(c) / n for c in product(range(n + 1), repeat=k) if sum(c) == n]

def search(names, step, folds=None):
    folds = list(FOLD_IDX) if folds is None else folds
    P = [MEMBERS[n]['oof'] for n in names]
    best_w, best_s = None, -1
    for w in simplex(len(names), step):
        p = sum(wi * pi for wi, pi in zip(w, P))
        s = np.mean([average_precision_score(Y['DevSet1010'][FOLD_IDX[f]], p[FOLD_IDX[f]]) for f in folds])
        if s > best_s + 1e-12:
            best_w, best_s = w, s
    return best_w, best_s

def crossfit(names, step):
    P = [MEMBERS[n]['oof'] for n in names]
    sc = []
    for f in FOLD_IDX:
        w, _ = search(names, step, [g for g in FOLD_IDX if g != f])
        p = sum(wi * pi for wi, pi in zip(w, P))
        sc.append(average_precision_score(Y['DevSet1010'][FOLD_IDX[f]], p[FOLD_IDX[f]]))
    return float(np.mean(sc))

SPECS = [('C0_current', ['V1_s0', 'MS_s0'], 0.05)]
if BEST_MOD:
    SPECS.append(('C1_newmod_pair', [f'{PAIR[0]}_s0', f'{PAIR[1]}_s0'], 0.05))
SPECS.append(('C2_bagged_pair', [f'{PAIR[0]}_bag3', f'{PAIR[1]}_bag3'], 0.05))
if BEST_MOD:
    SPECS.append(('C3_union', ['V1_s0', 'MS_s0', f'{PAIR[0]}_bag3', f'{PAIR[1]}_bag3'], 0.1))

rows = []
for cname, names, step in SPECS:
    w, s_in = search(names, step)
    s_cf = crossfit(names, step)
    if cname == 'C0_current':
        w, s_in = np.array([w0, 1 - w0]), fold_mean(C0['oof'])   # keep the original final exactly
    else:
        CANDS[cname] = combo(names, w)
    rows.append({'candidate': cname, 'members': ' + '.join(names), 'weights': np.round(w, 3).tolist(),
                 'Dev_in_sample': s_in, 'Dev_crossfit': s_cf})
dec = pd.DataFrame(rows)
win = dec.iloc[0]
for _, r in dec.iloc[1:].iterrows():
    if r.Dev_crossfit - win.Dev_crossfit >= MARGIN:
        win = r
dec['crossfit_gain_vs_C0'] = dec.Dev_crossfit - dec.Dev_crossfit.iloc[0]
dec.to_csv(V3_ROOT/'CANDIDATES_DEV.csv', index=False)
display(dec.round(4))
WINNER = str(win.candidate)
print('DEV-ONLY DECISION:', WINNER)

BOOT_C = []
for cname in CANDS:
    if cname == 'C0_current':
        continue
    print('\n' + cname)
    tab = compare('C0_current', C0, cname, CANDS[cname])
    BOOT_C.append(tab)
    show_auprc(tab)
BOOT_ALL = pd.concat([C0_CI, *BOOT_A, *BOOT_C], ignore_index=True)
BOOT_ALL.to_csv(V3_ROOT/'BOOTSTRAP_ALL_B10000.csv', index=False)

summary = []
for cname, c in CANDS.items():
    summary.append({'candidate': cname, 'Dev_fold_mean': fold_mean(c['oof']),
                    **{f'{s}_AUPRC': average_precision_score(Y[s], c[s]) for s in SPLITS_EXT},
                    'POOLED_AUPRC': average_precision_score(_pooled(Y['TestSet300'], Y['TestSetNew46']),
                                                            _pooled(c['TestSet300'], c['TestSetNew46'])),
                    'Dev_MCC_threshold': c['thr']})
summary = pd.DataFrame(summary)
summary.to_csv(V3_ROOT/'CANDIDATES_EXTERNAL.csv', index=False)
display(summary.round(4))

final = WINNER
if WINNER != 'C0_current':
    t = [b for b in BOOT_C if b.new_model.iloc[0] == WINNER][0]
    pr = t[(t.dataset == 'POOLED_TEST') & (t.metric == 'AUPRC')].iloc[0]
    print(f"\n{WINNER} vs C0, pooled test AUPRC: {pr.delta:+.4f} CI95 [{pr.delta_CI95_lo:+.4f}, {pr.delta_CI95_hi:+.4f}] "
          f"P={pr['P(new>ref)']:.3f}")
    if pr.delta_CI95_hi < 0:
        final = 'C0_current'
        print('STOP RULE: significantly worse on the tests -> keep the current final')
    elif pr.delta_CI95_lo > 0:
        print('VERDICT: significantly better on the tests -> new final =', WINNER)
    else:
        print('VERDICT: not significant on the tests -> the Dev decision stands: new final =', WINNER)
else:
    print('VERDICT: nothing beats the current final on Dev -> keep C0')
FINAL = CANDS[final]
(V3_ROOT/'FINAL_DECISION.json').write_text(json.dumps({
    'task': V3['TASK'], 'final': final, 'dev_only_winner': WINNER, 'members': FINAL['members'], 'weights': FINAL['weights'],
    'dev_mcc_threshold': FINAL['thr'], 'best_modality': list(BEST_MOD), 'bootstrap_B': B_BOOT,
    'candidates_dev': dec.to_dict('records')}, indent=2, default=str))
print('FINAL:', final, FINAL['members'], FINAL['weights'])


,candidate,members,weights,Dev_in_sample,Dev_crossfit,crossfit_gain_vs_C0
0,C0_current,V1_s0 + MS_s0,"[0.6, 0.4]",0.4093,0.4089,0.0000
1,C1_newmod_pair,V1+ESMIF_EF+PROTT5_s0 + MS+ESMIF_EF+PROTT5_s0,"[0.45, 0.55]",0.4743,0.4741,0.0652
2,C2_bagged_pair,V1+ESMIF_EF+PROTT5_bag3 + MS+ESMIF_EF+PROTT5_bag3,"[0.5, 0.5]",0.4829,0.4825,0.0737
3,C3_union,V1_s0 + MS_s0 + V1+ESMIF_EF+PROTT5_bag3 + MS+E...,"[0.1, 0.1, 0.4, 0.4]",0.4865,0.4852,0.0763


DEV-ONLY DECISION: C3_union

C1_newmod_pair
  bootstrap C1_newmod_pair vs C0_current | DevSet1010: 208s
  bootstrap C1_newmod_pair vs C0_current | TestSet300: 74s
  bootstrap C1_newmod_pair vs C0_current | TestSetNew46: 8s
  bootstrap C1_newmod_pair vs C0_current | POOLED_TEST: 81s


,dataset,ref,new,delta,delta_CI95_lo,delta_CI95_hi,P(new>ref)
0,DevSet1010,0.3950,0.4686,0.0735,0.0557,0.0905,1.0000
4,TestSet300,0.4460,0.4961,0.0501,0.0329,0.0671,1.0000
8,TestSetNew46,0.4274,0.4671,0.0397,0.0044,0.0762,0.9826
12,POOLED_TEST,0.4439,0.4935,0.0496,0.0330,0.0654,1.0000



C2_bagged_pair
  bootstrap C2_bagged_pair vs C0_current | DevSet1010: 206s
  bootstrap C2_bagged_pair vs C0_current | TestSet300: 74s
  bootstrap C2_bagged_pair vs C0_current | TestSetNew46: 8s
  bootstrap C2_bagged_pair vs C0_current | POOLED_TEST: 81s


,dataset,ref,new,delta,delta_CI95_lo,delta_CI95_hi,P(new>ref)
0,DevSet1010,0.3950,0.4820,0.0870,0.0702,0.1032,1.0000
4,TestSet300,0.4460,0.4992,0.0532,0.0358,0.0705,1.0000
8,TestSetNew46,0.4274,0.4710,0.0436,0.0098,0.0768,0.9923
12,POOLED_TEST,0.4439,0.4967,0.0529,0.0360,0.0688,1.0000



C3_union
  bootstrap C3_union vs C0_current | DevSet1010: 204s
  bootstrap C3_union vs C0_current | TestSet300: 73s
  bootstrap C3_union vs C0_current | TestSetNew46: 8s
  bootstrap C3_union vs C0_current | POOLED_TEST: 82s


,dataset,ref,new,delta,delta_CI95_lo,delta_CI95_hi,P(new>ref)
0,DevSet1010,0.3950,0.4851,0.0901,0.0766,0.1030,1.0000
4,TestSet300,0.4460,0.5041,0.0581,0.0447,0.0716,1.0000
8,TestSetNew46,0.4274,0.4716,0.0442,0.0186,0.0711,0.9998
12,POOLED_TEST,0.4439,0.5010,0.0571,0.0444,0.0695,1.0000


,candidate,Dev_fold_mean,TestSet300_AUPRC,TestSetNew46_AUPRC,POOLED_AUPRC,Dev_MCC_threshold
0,C0_current,0.4093,0.4460,0.4274,0.4439,0.5148
1,C1_newmod_pair,0.4743,0.4961,0.4671,0.4935,0.5693
2,C2_bagged_pair,0.4829,0.4992,0.4710,0.4967,0.5594
3,C3_union,0.4865,0.5041,0.4716,0.5010,0.4901



C3_union vs C0, pooled test AUPRC: +0.0571 CI95 [+0.0444, +0.0695] P=1.000
VERDICT: significantly better on the tests -> new final = C3_union
FINAL: C3_union ['V1_s0', 'MS_s0', 'V1+ESMIF_EF+PROTT5_bag3', 'MS+ESMIF_EF+PROTT5_bag3'] [0.1, 0.1, 0.4, 0.4]


In [30]:
# ============================================================
# V3-7. Export -> download, unzip into protein-binding/data/ (paths relative to Protein-binding-Phi/)
# ============================================================
import zipfile
OUT = BASE/'EXPORT'
OUT.mkdir(exist_ok=True)
zp = OUT/f"{V3['TASK'].lower()}_v3_improve.zip"
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(V3_ROOT.rglob('*')):   # member predictions, new member states, bootstrap + decision tables
        if p.is_file():
            z.write(p, p.relative_to(BASE))
print('zip:', zp, f'{zp.stat().st_size/1e6:.1f} MB')
EXPORT_PROTT5 = 'PROTT5' in BEST_MOD   # ProtT5 features are only needed locally when the final model uses them
if EXPORT_PROTT5:
    zf = OUT/'prott5_xl_half_v1_features.zip'
    if not zf.exists():
        with zipfile.ZipFile(zf, 'w', zipfile.ZIP_STORED) as z:
            for p in sorted(PROTT5_ROOT.rglob('*.npy')):
                z.write(p, p.relative_to(BASE))
    print('zip:', zf, f'{zf.stat().st_size/1e6:.1f} MB')


zip: /content/drive/MyDrive/Protein-binding-Phi/EXPORT/small_v3_improve.zip 148.3 MB
zip: /content/drive/MyDrive/Protein-binding-Phi/EXPORT/prott5_xl_half_v1_features.zip 490.1 MB
